**HSV_ablation**

In [4]:
%%writefile train_swin_three_models.py
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import re
import csv
import json
import time
import math
import shutil
import random
import argparse
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List, ContextManager, Union
from collections import Counter
from contextlib import nullcontext

import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.datasets import ImageFolder

import timm
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)


# -------------------------
# Utilities
# -------------------------
def set_seed(seed: int, deterministic: bool = True) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.backends.cuda.matmul.allow_tf32 = False
            torch.backends.cudnn.allow_tf32 = False
        except Exception:
            pass

        try:
            torch.use_deterministic_algorithms(True)
        except Exception as e:
            warnings.warn(
                f"Could not enable deterministic algorithms: {e}\n"
                "Training will continue with partial determinism."
            )
    else:
        torch.backends.cudnn.benchmark = True


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_autocast_ctx(use_amp: bool) -> ContextManager:
    enabled = bool(use_amp and torch.cuda.is_available())
    if not enabled:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return torch.cuda.amp.autocast(enabled=True)


def make_grad_scaler(use_amp: bool):
    enabled = bool(use_amp and torch.cuda.is_available())
    try:
        return torch.cuda.amp.GradScaler(enabled=enabled)
    except Exception:
        if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
            return torch.amp.GradScaler(enabled=enabled)
        raise


def rgb_to_hsv_torch(rgb01: torch.Tensor) -> torch.Tensor:
    r, g, b = rgb01[:, 0:1], rgb01[:, 1:2], rgb01[:, 2:3]
    maxc, _ = rgb01.max(dim=1, keepdim=True)
    minc, _ = rgb01.min(dim=1, keepdim=True)
    v = maxc
    delta = maxc - minc
    
    eps = 1e-10
    
    # Saturation
    s = delta / torch.clamp(maxc, min=eps)
    s = torch.where(maxc > eps, s, torch.zeros_like(s))
    
    # Hue computation with deterministic max-channel selection
    delta_safe = torch.clamp(delta, min=eps)
    
    # Compute hue formulas for each channel (standard HSV conversion)
    h_r = (g - b) / delta_safe          # If R is max
    h_g = (b - r) / delta_safe + 2.0    # If G is max
    h_b = (r - g) / delta_safe + 4.0    # If B is max
    
    idx = rgb01.argmax(dim=1, keepdim=True)   # [B,1,H,W] ∈ {0,1,2}
    mask = (delta > eps)                     # [B,1,H,W]

    h = torch.zeros_like(delta, dtype=rgb01.dtype)  # [B,1,H,W]
    h = torch.where((idx == 0) & mask, h_r, h)
    h = torch.where((idx == 1) & mask, h_g, h)
    h = torch.where((idx == 2) & mask, h_b, h)

    # Normalize to [0, 1) with true modulo (handles wraparound correctly)
    h = torch.remainder(h / 6.0, 1.0)
    hsv = torch.cat([h, s, v], dim=1)
    return torch.clamp(hsv, 0.0, 1.0)


def hsv_to_sincos_sv(hsv01: torch.Tensor) -> torch.Tensor:
    """
    HSV [0,1] -> [sin(2πH), cos(2πH), S, V]
    Removes hue discontinuity at wrap-around.
    """
    h = hsv01[:, 0:1]
    s = hsv01[:, 1:2]
    v = hsv01[:, 2:3]
    ang = 2.0 * math.pi * h
    hsin = torch.sin(ang)
    hcos = torch.cos(ang)
    return torch.cat([hsin, hcos, s, v], dim=1)


def normalize_hsv_rep(x: torch.Tensor) -> torch.Tensor:
    """
    x: [hsin, hcos, s, v]
    Keep sin/cos as-is in [-1,1].
    Normalize S,V to roughly zero-mean and stable scale.
    """
    hsin = x[:, 0:1]
    hcos = x[:, 1:2]
    s = (x[:, 2:3] - 0.5) / 0.25
    v = (x[:, 3:4] - 0.5) / 0.25
    return torch.cat([hsin, hcos, s, v], dim=1)


def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


def try_get_gflops(model: nn.Module, input_size: int, device: str) -> Optional[float]:
    """GFLOPs via ptflops (optional). Returns None if unavailable."""
    try:
        from ptflops import get_model_complexity_info
    except Exception:
        return None

    if isinstance(model, nn.DataParallel):
        model = model.module

    model = model.to(device)
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            macs, _params = get_model_complexity_info(
                model, (3, input_size, input_size),
                as_strings=False,
                print_per_layer_stat=False,
                verbose=False
            )
        flops = 2.0 * float(macs)  # FLOPs ≈ 2*MACs
        return flops / 1e9
    except Exception as e:
        print(f"⚠️ GFLOPs computation failed: {e}")
        return None
    finally:
        if was_training:
            model.train()


def _json_safe_scalar(x):
    if x is None:
        return None
    if isinstance(x, (int, float, str, bool)):
        return x
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        val = float(x)
        if np.isnan(val) or np.isinf(val):
            return None
        return val
    if isinstance(x, torch.Tensor):
        if x.ndim == 0:
            return _json_safe_scalar(x.item())
        return x.detach().cpu().numpy().tolist()
    if isinstance(x, np.ndarray):
        if x.ndim == 0:
            return _json_safe_scalar(x.item())
        return x.tolist()
    return str(x)


def strip_module_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            out[k[7:]] = v
        else:
            out[k] = v
    return out


def add_module_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in state_dict.items():
        out[k if k.startswith("module.") else f"module.{k}"] = v
    return out


def load_state_dict_robust(model: nn.Module, state_dict: Dict[str, torch.Tensor]) -> None:
    """
    Robust load:
    - strict=True with (as-is / strip module / add module)
    - strict=False fallback with warnings
    """
    prefix_fns = [lambda x: x, strip_module_prefix, add_module_prefix]

    for fn in prefix_fns:
        try:
            model.load_state_dict(fn(state_dict), strict=True)
            return
        except Exception:
            pass

    for fn in prefix_fns:
        try:
            result = model.load_state_dict(fn(state_dict), strict=False)
            if getattr(result, "missing_keys", None) or getattr(result, "unexpected_keys", None):
                warnings.warn(
                    "Loaded with strict=False:\n"
                    f"  Missing keys: {result.missing_keys}\n"
                    f"  Unexpected keys: {result.unexpected_keys}"
                )
            return
        except Exception:
            pass

    raise RuntimeError("Could not load state dict with any strategy.")


# -------------------------
# EMA
# -------------------------
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.model = model
        self.decay = decay
        self.shadow: Dict[str, torch.Tensor] = {}
        self.backup: Dict[str, torch.Tensor] = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.data.clone()

    def update(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad:
                if name not in self.shadow:
                    self.shadow[name] = p.data.clone()
                else:
                    self.shadow[name] = (1.0 - self.decay) * p.data + self.decay * self.shadow[name]

    def apply_shadow(self):
        self.backup = {}
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.shadow:
                self.backup[name] = p.data.clone()
                p.data = self.shadow[name].clone()

    def restore(self):
        if not self.backup:
            warnings.warn("EMA.restore() called but backup is empty. Did you call apply_shadow()?")  # harmless
            return
        for name, p in self.model.named_parameters():
            if p.requires_grad and name in self.backup:
                p.data = self.backup[name].clone()
        self.backup = {}


# -------------------------
# Config
# -------------------------
@dataclass
class Config:
    # Model
    model_name: str = "swin_tiny_patch4_window7_224"
    num_classes: int = 7
    pretrained: bool = False
    drop_path_rate: float = 0.2

    # Data
    data_root: str = "/kaggle/input/tea-leaf701515/tea_leaf_processed_dataset"
    input_size: int = 224
    batch_size: int = 64
    num_workers: int = 2

    # ImageNet stats
    img_mean: Tuple[float, float, float] = (0.485, 0.456, 0.406)
    img_std: Tuple[float, float, float] = (0.229, 0.224, 0.225)

    # HSV stabilization
    hsv_use_sincos: bool = True     # use sin/cos hue representation (recommended)
    gate_warmup_epochs: int = 5     # gate_alpha ramps 0->1 over these epochs

    # Training
    epochs: int = 80
    warmup_epochs: int = 5
    lr: float = 5e-4
    warmup_lr_init: float = 1e-6
    weight_decay: float = 0.05
    min_lr: float = 1e-6
    label_smoothing: float = 0.1
    grad_clip_norm: float = 1.0
    gradient_accumulation_steps: int = 1

    # Augmentation
    color_jitter: float = 0.2
    hue_jitter: float = 0.1
    use_gaussian_blur: bool = False
    gaussian_blur_prob: float = 0.1
    rrc_scale_min: float = 0.85

    # Optimization
    use_amp: bool = True
    use_ema: bool = False
    ema_decay: float = 0.9998
    use_class_weights: bool = False

    # HSV branch
    use_hsv_branch: bool = False
    hsv_embed_dim: int = 128
    hsv_dropout: float = 0.1
    fuse_dropout: float = 0.2
    gate_hidden: int = 256
    gate_vector: bool = False

    # Evaluation
    compute_val_auc: bool = False
    early_stopping_patience: int = 25

    # Logging / folders
    run_dir: str = "/kaggle/working/experiments_swin"
    experiment_name: str = "rgb_baseline"
    log_interval: int = 50
    save_cm_png: bool = True

    # Checkpointing
    ckpt_temp_dir: str = "/kaggle/temp"
    save_epoch_checkpoints: bool = False
    save_epoch_every: int = 5
    keep_last_n_checkpoints: int = 3

    # System
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    deterministic: bool = True
    use_data_parallel: bool = True

    def validate(self):
        invalid_chars = ['/', '\\', '..', '\0']
        if any(char in self.experiment_name for char in invalid_chars):
            raise ValueError(f"Invalid experiment_name: {self.experiment_name}")
        if not Path(self.data_root).exists():
            raise FileNotFoundError(f"data_root does not exist: {self.data_root}")
        if self.batch_size < 1:
            raise ValueError("batch_size must be >= 1")
        if self.epochs < 1:
            raise ValueError("epochs must be >= 1")
        if not 0 <= self.label_smoothing < 1:
            raise ValueError("label_smoothing must be in [0,1)")
        if self.gradient_accumulation_steps < 1:
            raise ValueError("gradient_accumulation_steps must be >= 1")
        if self.warmup_epochs < 0:
            raise ValueError("warmup_epochs must be >= 0")
        if self.lr <= 0:
            raise ValueError("lr must be > 0")
        if not (0.0 <= self.ema_decay <= 1.0):
            raise ValueError("ema_decay must be in [0,1]")
        if self.drop_path_rate < 0:
            raise ValueError("drop_path_rate must be >= 0")


# -------------------------
# Model
# -------------------------
class SwinRGBHSV(nn.Module):
    def __init__(
        self,
        model_name: str,
        pretrained: bool,
        drop_path_rate: float,
        num_classes: int,
        use_hsv_branch: bool,
        hsv_embed_dim: int = 128,
        hsv_use_sincos: bool = True,
        hsv_dropout: float = 0.1,
        fuse_dropout: float = 0.2,
        gate_hidden: int = 256,
        gate_vector: bool = False,
        img_mean: Tuple[float, float, float] = (0.485, 0.456, 0.406),
        img_std: Tuple[float, float, float] = (0.229, 0.224, 0.225),
    ):
        super().__init__()
        self.use_hsv_branch = use_hsv_branch
        self.hsv_use_sincos = hsv_use_sincos
        self.gate_vector = gate_vector

        self.register_buffer("img_mean", torch.tensor(img_mean).view(1, 3, 1, 1))
        self.register_buffer("img_std", torch.tensor(img_std).view(1, 3, 1, 1))

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            drop_path_rate=drop_path_rate,
            global_pool="avg",
        )
        feat_dim = getattr(self.backbone, "num_features", 768)
        self.feat_dim = int(feat_dim)

        if self.use_hsv_branch:
            in_ch = 4 if self.hsv_use_sincos else 3

            self.hsv_branch = nn.Sequential(
                nn.Conv2d(in_ch, 16, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(16),
                nn.GELU(),
                nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.GELU(),
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(p=hsv_dropout),
                nn.Linear(32, hsv_embed_dim),
                nn.GELU(),
            )

            self.hsv_proj = nn.Sequential(
                nn.Dropout(p=fuse_dropout),
                nn.Linear(hsv_embed_dim, self.feat_dim),
            )

            gate_out_dim = self.feat_dim if self.gate_vector else 1
            self.gate_mlp = nn.Sequential(
                nn.Linear(self.feat_dim + hsv_embed_dim, gate_hidden),
                nn.GELU(),
                nn.Dropout(p=fuse_dropout),
                nn.Linear(gate_hidden, gate_out_dim),
            )
            if self.gate_mlp[-1].bias is not None:
                nn.init.constant_(self.gate_mlp[-1].bias, -2.0)

            self.classifier = nn.Sequential(
                nn.Dropout(p=fuse_dropout),
                nn.Linear(self.feat_dim, self.feat_dim // 2),
                nn.GELU(),
                nn.Dropout(p=fuse_dropout),
                nn.Linear(self.feat_dim // 2, num_classes),
            )
        else:
            self.classifier = nn.Linear(self.feat_dim, num_classes)

    def forward(self, x_norm: torch.Tensor, return_gate: bool = False, gate_alpha: float = 1.0):
        feat = self.backbone(x_norm)

        # RGB-only path
        if not self.use_hsv_branch:
            logits = self.classifier(feat)
            if return_gate:
                gate = torch.zeros((logits.size(0), 1), device=logits.device, dtype=logits.dtype)
                return logits, gate
            return logits

        # --- Force FP32 for HSV conversion (stable under AMP) ---
        # HSV conversion in pure float32, regardless of autocast.
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast(enabled=False):
                x_rgb01 = (x_norm.float() * self.img_std.float()) + self.img_mean.float()
                x_rgb01 = torch.clamp(x_rgb01, 0.0, 1.0)

                x_hsv = rgb_to_hsv_torch(x_rgb01)
                if self.hsv_use_sincos:
                    x_hsv_rep = hsv_to_sincos_sv(x_hsv)
                    x_hsv_rep = normalize_hsv_rep(x_hsv_rep)
                else:
                    x_hsv_rep = x_hsv  # raw HSV [0,1]
        else:
            # CPU fallback (still float32)
            x_rgb01 = (x_norm.float() * self.img_std.float()) + self.img_mean.float()
            x_rgb01 = torch.clamp(x_rgb01, 0.0, 1.0)

            x_hsv = rgb_to_hsv_torch(x_rgb01)
            if self.hsv_use_sincos:
                x_hsv_rep = hsv_to_sincos_sv(x_hsv)
                x_hsv_rep = normalize_hsv_rep(x_hsv_rep)
            else:
                x_hsv_rep = x_hsv  # raw HSV [0,1]

        # Match dtype for conv/bn path under AMP (usually fp16/bf16)
        x_hsv_rep = x_hsv_rep.to(dtype=feat.dtype)

        hsv_emb = self.hsv_branch(x_hsv_rep)

        # Gate in float32 for stable sigmoid / MLP numerics
        gate_in = torch.cat([feat.float(), hsv_emb.float()], dim=1)
        gate = torch.sigmoid(self.gate_mlp(gate_in))

        if not self.gate_vector:
            gate_for_fuse = gate.expand(-1, self.feat_dim)
            gate_stat = gate
        else:
            gate_for_fuse = gate
            gate_stat = gate.mean(dim=1, keepdim=True)

        hsv_feat = self.hsv_proj(hsv_emb)

        # Fuse in backbone dtype
        gate_for_fuse = gate_for_fuse.to(dtype=feat.dtype)
        hsv_feat = hsv_feat.to(dtype=feat.dtype)

        alpha = float(gate_alpha)
        fused = feat + (alpha * gate_for_fuse) * hsv_feat
        logits = self.classifier(fused)

        if return_gate:
            return logits, gate_stat
        return logits

# -------------------------
# Checkpoint Manager
# -------------------------
class CheckpointManager:
    def __init__(self, temp_root: Path, final_root: Path, experiment_name: str):
        self.temp_dir = temp_root / experiment_name
        self.final_dir = final_root / experiment_name
        self.pid = os.getpid()

        self.temp_dir.mkdir(parents=True, exist_ok=True)

        (self.final_dir / "model").mkdir(parents=True, exist_ok=True)
        (self.final_dir / "config").mkdir(parents=True, exist_ok=True)
        (self.final_dir / "logs").mkdir(parents=True, exist_ok=True)
        (self.final_dir / "metrics").mkdir(parents=True, exist_ok=True)
        (self.final_dir / "raw_outputs").mkdir(parents=True, exist_ok=True)
        (self.final_dir / "figures").mkdir(parents=True, exist_ok=True)

        print(f"\n📁 Directories:")
        print(f"   Temp:  {self.temp_dir}")
        print(f"   Final: {self.final_dir}")

    def save_checkpoint(
        self,
        checkpoint: Dict,
        epoch: int,
        is_best: bool,
        save_epoch_ckpt: bool = False,
        epoch_interval: int = 5,
        keep_n: int = 3
    ) -> None:
        def _atomic_save(obj, final_path: Path) -> bool:
            tmp_path = final_path.with_suffix(f".{self.pid}.tmp")
            try:
                torch.save(obj, tmp_path)
                tmp_path.replace(final_path)
                return True
            except Exception as e:
                try:
                    if tmp_path.exists():
                        tmp_path.unlink()
                except Exception:
                    pass
                print(f"⚠️ Checkpoint save failed ({final_path.name}): {repr(e)}")
                return False

        _atomic_save(checkpoint, self.temp_dir / "last_checkpoint.pth")
        if is_best:
            _atomic_save(checkpoint, self.temp_dir / "best_model.pth")

        if save_epoch_ckpt and (not is_best) and (epoch % max(1, epoch_interval) == 0):
            ckpt_path = self.temp_dir / f"checkpoint_epoch_{epoch}.pth"
            if _atomic_save(checkpoint, ckpt_path):
                self._cleanup_old_epochs(keep_n)

    def _cleanup_old_epochs(self, keep_n: int) -> None:
        epoch_ckpts = sorted(
            self.temp_dir.glob("checkpoint_epoch_*.pth"),
            key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.name).group(1))
            if re.search(r"checkpoint_epoch_(\d+)", p.name) else 0
        )
        if len(epoch_ckpts) <= keep_n:
            return
        for old in epoch_ckpts[:-keep_n]:
            try:
                old.unlink()
            except Exception:
                pass

    def _save_confusion_matrix_png(self, cm: np.ndarray, classes: List[str], out_path: Path) -> None:
        try:
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(8, 8))
            im = ax.imshow(cm, interpolation="nearest")
            ax.figure.colorbar(im, ax=ax)
            ax.set(
                xticks=np.arange(len(classes)),
                yticks=np.arange(len(classes)),
                xticklabels=classes,
                yticklabels=classes,
                ylabel="True label",
                xlabel="Predicted label",
                title="Confusion Matrix"
            )
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

            thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    ax.text(
                        j, i, format(cm[i, j], "d"),
                        ha="center", va="center",
                        color="white" if cm[i, j] > thresh else "black"
                    )
            fig.tight_layout()
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)
        except Exception as e:
            print(f"⚠️ Could not save confusion_matrix.png: {e}")

    def copy_final_artifacts(
        self,
        best_ckpt_path: Path,
        config: Config,
        train_log: List[Dict],
        test_metrics: Dict,
        class_to_idx: Dict[str, int],
        classes: List[str],
        logs_dir: Path,
    ) -> None:
        print("\n" + "=" * 80)
        print("COPYING FINAL ARTIFACTS")
        print("=" * 80)

        model_dir = self.final_dir / "model"
        config_dir = self.final_dir / "config"
        logs_dst = self.final_dir / "logs"
        metrics_dir = self.final_dir / "metrics"
        raw_dir = self.final_dir / "raw_outputs"
        fig_dir = self.final_dir / "figures"

        if not best_ckpt_path.exists():
            raise FileNotFoundError(f"best_model.pth not found: {best_ckpt_path}")

        shutil.copy2(best_ckpt_path, model_dir / "best_model.pth")
        best_ckpt = torch.load(best_ckpt_path, map_location="cpu")

        torch.save(best_ckpt["model"], model_dir / "model_weights_only.pth")

        if "ema_shadow" in best_ckpt and isinstance(best_ckpt["ema_shadow"], dict):
            ema_state = dict(best_ckpt["model"])
            for k, v in best_ckpt["ema_shadow"].items():
                if k in ema_state:
                    ema_state[k] = v
            torch.save(ema_state, model_dir / "ema_model_weights_only.pth")

        with open(config_dir / "config.json", "w") as f:
            json.dump(config.__dict__, f, indent=2)
        with open(config_dir / "class_to_idx.json", "w") as f:
            json.dump(class_to_idx, f, indent=2)
        with open(config_dir / "classes.json", "w") as f:
            json.dump(classes, f, indent=2)

        if train_log:
            with open(logs_dst / "train_log.csv", "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=train_log[0].keys())
                writer.writeheader()
                writer.writerows(train_log)

        gate_src = Path(logs_dir) / "gate_stats_log.csv"
        gate_dst = logs_dst / "gate_stats_log.csv"
        if gate_src.exists():
            if gate_src.resolve() != gate_dst.resolve():
                shutil.copy2(gate_src, gate_dst)

        results = {}
        for k, v in test_metrics.items():
            if not isinstance(v, (np.ndarray, list, dict)):
                scalar_val = _json_safe_scalar(v)
                if k == "macro_auc" and scalar_val == -1.0:
                    results[k] = "N/A"
                else:
                    results[k] = scalar_val

        with open(metrics_dir / "test_results.json", "w") as f:
            json.dump(results, f, indent=2)

        if "targets" in test_metrics and "preds" in test_metrics:
            cm = confusion_matrix(test_metrics["targets"], test_metrics["preds"])
            np.save(metrics_dir / "confusion_matrix.npy", cm)

            rep = classification_report(
                test_metrics["targets"],
                test_metrics["preds"],
                target_names=classes,
                digits=4
            )
            with open(metrics_dir / "classification_report.txt", "w") as f:
                f.write(rep)

            per_class_f1 = f1_score(
                test_metrics["targets"], test_metrics["preds"],
                average=None, labels=list(range(len(classes)))
            )
            per_class_precision = precision_score(
                test_metrics["targets"], test_metrics["preds"],
                average=None, labels=list(range(len(classes))), zero_division=0
            )
            per_class_recall = recall_score(
                test_metrics["targets"], test_metrics["preds"],
                average=None, labels=list(range(len(classes))), zero_division=0
            )

            per_class_metrics = {}
            for i, cname in enumerate(classes):
                per_class_metrics[cname] = {
                    "f1": float(per_class_f1[i]),
                    "precision": float(per_class_precision[i]),
                    "recall": float(per_class_recall[i]),
                }
            with open(metrics_dir / "per_class_metrics.json", "w") as f:
                json.dump(per_class_metrics, f, indent=2)

            if config.save_cm_png:
                self._save_confusion_matrix_png(cm, classes, fig_dir / "confusion_matrix.png")

        if "preds" in test_metrics:
            np.save(raw_dir / "test_predictions.npy", test_metrics["preds"])
        if "targets" in test_metrics:
            np.save(raw_dir / "test_targets.npy", test_metrics["targets"])
        if "probs" in test_metrics:
            np.save(raw_dir / "test_probabilities.npy", test_metrics["probs"])
        if "gate_stats" in test_metrics:
            with open(raw_dir / "gate_stats_test.json", "w") as f:
                json.dump(test_metrics["gate_stats"], f, indent=2)

        print(f"✅ Final bundle ready at: {self.final_dir}")


# -------------------------
# Trainer
# -------------------------
class Trainer:
    def __init__(self, cfg: Config):
        cfg.validate()
        self.cfg = cfg
        set_seed(cfg.seed, deterministic=cfg.deterministic)

        if cfg.deterministic and cfg.num_workers > 0:
            print(
                "⚠️ Determinism note: num_workers > 0 with random augmentations may still introduce "
                "small non-determinism. For maximum determinism, set --num_workers 0."
            )

        self.run_root = Path(cfg.run_dir)
        self.run_root.mkdir(parents=True, exist_ok=True)

        self.exp_dir = self.run_root / cfg.experiment_name
        self.exp_dir.mkdir(parents=True, exist_ok=True)

        self.logs_dir = self.exp_dir / "logs"
        self.logs_dir.mkdir(exist_ok=True)
        self.config_dir = self.exp_dir / "config"
        self.config_dir.mkdir(exist_ok=True)

        self.ckpt_manager = CheckpointManager(
            temp_root=Path(cfg.ckpt_temp_dir),
            final_root=self.run_root,
            experiment_name=cfg.experiment_name
        )

        self.model = self._build_model().to(cfg.device)

        self.params_m = float(count_params_m(self.model))
        self.gflops = try_get_gflops(self.model, cfg.input_size, cfg.device)
        self.gflops_out = float(self.gflops) if self.gflops is not None else -1.0

        print(f"   Params: {self.params_m:.2f}M")
        print(f"   GFLOPs: {self.gflops_out:.2f}" if self.gflops_out >= 0 else "   GFLOPs: N/A")

        self.n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
        if self.n_gpus > 1 and cfg.use_data_parallel:
            print(f"\n🚀 DataParallel: {self.n_gpus} GPUs")
            self.model = nn.DataParallel(self.model)
        else:
            if torch.cuda.is_available():
                print(f"\n💻 Single GPU: {torch.cuda.get_device_name(0)}")
            self.n_gpus = 1

        base_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        self.ema = EMA(base_model, cfg.ema_decay) if cfg.use_ema else None

        self.train_loader, self.val_loader, self.test_loader = self._build_loaders()

        self.class_weights = None
        if cfg.use_class_weights:
            class_counts = Counter(self.train_loader.dataset.targets)
            weights = []
            missing = []
            for i in range(cfg.num_classes):
                c = int(class_counts.get(i, 0))
                if c == 0:
                    missing.append(i)
                weights.append(1.0 / max(c, 1))

            if missing:
                warnings.warn(
                    f"Some classes are missing in TRAIN split: {missing}. "
                    "Using weight=1.0 for them."
                )

            mean_w = sum(weights) / max(1, len(weights))
            weights = [w / max(mean_w, 1e-12) for w in weights]

            self.class_weights = torch.tensor(weights, dtype=torch.float32, device=cfg.device)
            print(f"\n⚖️  Using class weights: {[f'{w:.3f}' for w in weights]}")

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=cfg.lr,
            weight_decay=cfg.weight_decay,
            betas=(0.9, 0.999),
        )

        batches_per_epoch = len(self.train_loader)
        accum = int(cfg.gradient_accumulation_steps)
        updates_per_epoch = int(math.ceil(batches_per_epoch / accum))
        total_updates = int(cfg.epochs * updates_per_epoch)
        warmup_updates = int(cfg.warmup_epochs * updates_per_epoch)
        warmup_updates = min(warmup_updates, max(0, total_updates - 1))

        print("\n📅 Scheduler Setup (optimizer updates):")
        print(f"   batches/epoch: {batches_per_epoch}")
        print(f"   accum_steps:   {accum}")
        print(f"   updates/epoch: {updates_per_epoch}")
        print(f"   warmup_updates:{warmup_updates}")
        print(f"   total_updates: {total_updates}")

        if warmup_updates > 0:
            warmup_scheduler = LinearLR(
                self.optimizer,
                start_factor=cfg.warmup_lr_init / cfg.lr,
                end_factor=1.0,
                total_iters=warmup_updates,
            )
            cosine_updates = max(1, total_updates - warmup_updates)
            cosine_scheduler = CosineAnnealingLR(
                self.optimizer,
                T_max=cosine_updates,
                eta_min=cfg.min_lr,
            )
            self.scheduler = SequentialLR(
                self.optimizer,
                schedulers=[warmup_scheduler, cosine_scheduler],
                milestones=[warmup_updates],
            )
        else:
            self.scheduler = CosineAnnealingLR(
                self.optimizer,
                T_max=max(1, total_updates),
                eta_min=cfg.min_lr,
            )

        self.scaler = make_grad_scaler(cfg.use_amp)

        self.best_val_f1 = -1.0
        self.bad_epochs = 0
        self.start_epoch = 1
        self.train_log: List[Dict[str, float]] = []

        self._save_config()
        self._save_env()
        self._validate_dataset()

    def _save_config(self):
        with open(self.config_dir / "config.json", "w") as f:
            json.dump(self.cfg.__dict__, f, indent=2)

    def _save_env(self):
        info = {
            "python": os.popen("python -V").read().strip(),
            "torch": torch.__version__,
            "timm": timm.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
            "cudnn_version": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
            "gpu0": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "num_gpus": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        }
        with open(self.config_dir / "env.json", "w") as f:
            json.dump(info, f, indent=2)

    def _validate_dataset(self):
        print("\n🔍 Dataset Validation:")
        train_targets = self.train_loader.dataset.targets
        val_targets = self.val_loader.dataset.targets
        test_targets = self.test_loader.dataset.targets

        train_dist = Counter(train_targets)
        val_dist = Counter(val_targets)
        test_dist = Counter(test_targets)

        class_names = self.train_loader.dataset.classes
        print("\n  Class Distribution:")
        counts = []
        for ci, cname in enumerate(class_names):
            tr = train_dist.get(ci, 0)
            va = val_dist.get(ci, 0)
            te = test_dist.get(ci, 0)
            print(f"    {cname:20s} -> Train: {tr:5d} | Val: {va:5d} | Test: {te:5d}")
            counts.append(tr)
            if tr < 10:
                print(f"    ⚠️ Very low train samples for class '{cname}': {tr}")

        if len(counts) > 0 and min(counts) > 0:
            ratio = max(counts) / min(counts)
            if ratio > 10:
                print(f"\n  ⚠️  Severe imbalance ratio: {ratio:.1f}:1 (max/min train class)")
                print(f"      Consider: class weighting, focal loss, or resampling")
            elif ratio > 3:
                print(f"\n  ℹ️  Moderate imbalance ratio: {ratio:.1f}:1")

            median_count = np.median(counts)
            for ci, cname in enumerate(class_names):
                if counts[ci] < median_count * 0.5:
                    print(
                        f"      ⚠️  Class '{cname}' underrepresented: "
                        f"{counts[ci]} samples (median: {int(median_count)})"
                    )
        print()

    def _build_model(self) -> nn.Module:
        print(f"\n🏗️  Building model: {self.cfg.model_name}")
        print(f"   Pretrained: {self.cfg.pretrained}")
        print(f"   DropPath:   {self.cfg.drop_path_rate}")
        print(f"   RGB+HSV:    {self.cfg.use_hsv_branch} (gate_vector={self.cfg.gate_vector})")
        print(f"   HSV sincos: {self.cfg.hsv_use_sincos} | gate warmup: {self.cfg.gate_warmup_epochs} ep")

        return SwinRGBHSV(
            model_name=self.cfg.model_name,
            pretrained=self.cfg.pretrained,
            drop_path_rate=self.cfg.drop_path_rate,
            num_classes=self.cfg.num_classes,
            use_hsv_branch=self.cfg.use_hsv_branch,
            hsv_embed_dim=self.cfg.hsv_embed_dim,
            hsv_use_sincos=self.cfg.hsv_use_sincos,
            hsv_dropout=self.cfg.hsv_dropout,
            fuse_dropout=self.cfg.fuse_dropout,
            gate_hidden=self.cfg.gate_hidden,
            gate_vector=self.cfg.gate_vector,
            img_mean=self.cfg.img_mean,
            img_std=self.cfg.img_std,
        )

    def _build_loaders(self) -> Tuple[DataLoader, DataLoader, DataLoader]:
        root = Path(self.cfg.data_root)
        for split in ["train", "val", "test"]:
            if not (root / split).exists():
                raise FileNotFoundError(f"Missing split folder: {root/split}")

        # IMPORTANT: hue jitter can destabilize HSV branch (circular hue).
        # Auto-disable hue jitter whenever HSV branch is enabled.
        hue_for_train = 0.0 if self.cfg.use_hsv_branch else self.cfg.hue_jitter

        train_tf_list = [
            T.RandomResizedCrop(
                self.cfg.input_size,
                scale=(self.cfg.rrc_scale_min, 1.0),
                interpolation=InterpolationMode.BICUBIC
            ),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(
                brightness=self.cfg.color_jitter,
                contrast=self.cfg.color_jitter,
                saturation=self.cfg.color_jitter,
                hue=hue_for_train
            ),
        ]
        if self.cfg.use_gaussian_blur:
            train_tf_list.append(
                T.RandomApply([T.GaussianBlur(kernel_size=3)], p=self.cfg.gaussian_blur_prob)
            )
        train_tf_list += [
            T.ToTensor(),
            T.Normalize(mean=self.cfg.img_mean, std=self.cfg.img_std),
        ]
        train_tf = T.Compose(train_tf_list)

        eval_tf = T.Compose([
            T.Resize(int(self.cfg.input_size * 1.14), interpolation=InterpolationMode.BICUBIC),
            T.CenterCrop(self.cfg.input_size),
            T.ToTensor(),
            T.Normalize(mean=self.cfg.img_mean, std=self.cfg.img_std),
        ])

        train_ds = ImageFolder(root / "train", transform=train_tf)
        val_ds = ImageFolder(root / "val", transform=eval_tf)
        test_ds = ImageFolder(root / "test", transform=eval_tf)

        if len(train_ds.classes) != self.cfg.num_classes:
            raise ValueError(
                f"num_classes={self.cfg.num_classes} but dataset has {len(train_ds.classes)} classes: {train_ds.classes}"
            )

        print(f"\n📦 Data: Train {len(train_ds)} | Val {len(val_ds)} | Test {len(test_ds)}")
        print(f"   Classes: {train_ds.classes}")
        if self.cfg.use_hsv_branch:
            print(f"   Aug note: hue jitter is DISABLED because HSV branch is enabled.")

        use_persistent = (self.cfg.num_workers > 0) and (self.cfg.epochs >= 2)

        train_loader = DataLoader(
            train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            num_workers=self.cfg.num_workers,
            pin_memory=True,
            drop_last=False,
            worker_init_fn=worker_init_fn,
            generator=torch.Generator().manual_seed(self.cfg.seed),
            persistent_workers=use_persistent,
        )
        val_loader = DataLoader(
            val_ds,
            batch_size=self.cfg.batch_size * 2,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            pin_memory=True,
            persistent_workers=use_persistent,
        )
        test_loader = DataLoader(
            test_ds,
            batch_size=self.cfg.batch_size * 2,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            pin_memory=True,
            persistent_workers=use_persistent,
        )
        return train_loader, val_loader, test_loader

    def _append_gate_csv(self, epoch: int, gate_stats: Dict[str, Dict[str, float]], gate_global_mean: float):
        gate_csv_path = self.logs_dir / "gate_stats_log.csv"
        row: Dict[str, Union[int, float]] = {"epoch": int(epoch), "gate_global_mean": float(gate_global_mean)}
        for cname, stats in gate_stats.items():
            row[f"{cname}_mean"] = float(stats["mean"])
            row[f"{cname}_std"] = float(stats["std"])
            row[f"{cname}_n"] = int(stats.get("n", 0))

        write_header = not gate_csv_path.exists()
        with open(gate_csv_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            if write_header:
                writer.writeheader()
            writer.writerow(row)

    def resume_from(self, ckpt_path: Path) -> None:
        print(f"\n📂 Resuming from: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=self.cfg.device)

        target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        load_state_dict_robust(target_model, ckpt["model"])

        if "optimizer" in ckpt:
            try:
                self.optimizer.load_state_dict(ckpt["optimizer"])
            except Exception as e:
                print(f"⚠️ Optimizer load failed: {e}")

        if "scheduler" in ckpt:
            try:
                self.scheduler.load_state_dict(ckpt["scheduler"])
            except Exception as e:
                print(f"⚠️ Scheduler load failed: {e}")

        if "scaler" in ckpt:
            try:
                self.scaler.load_state_dict(ckpt["scaler"])
            except Exception as e:
                print(f"⚠️ Scaler load failed: {e}")

        if self.ema is not None and "ema_shadow" in ckpt:
            try:
                for k, v in ckpt["ema_shadow"].items():
                    self.ema.shadow[k] = v.to(self.cfg.device, non_blocking=True)
                print("✅ EMA state loaded")
            except Exception as e:
                print(f"⚠️ EMA shadow load failed: {e}")

        self.best_val_f1 = float(ckpt.get("best_val_f1", self.best_val_f1))
        self.bad_epochs = int(ckpt.get("bad_epochs", self.bad_epochs))
        self.start_epoch = max(1, int(ckpt.get("epoch", 0)) + 1)

        if "train_log" in ckpt and isinstance(ckpt["train_log"], list):
            self.train_log = ckpt["train_log"]

        print(f"✅ Resumed: next epoch {self.start_epoch}, best F1 {self.best_val_f1:.4f}")

    def _gate_alpha_for_epoch(self, epoch: int) -> float:
        if not self.cfg.use_hsv_branch:
            return 1.0
        w = max(1, int(self.cfg.gate_warmup_epochs))
        return float(min(1.0, epoch / w))

    def train_one_epoch(self, epoch: int) -> Dict[str, float]:
        self.model.train()
        total_loss, correct, n = 0.0, 0, 0
        accum_steps = int(self.cfg.gradient_accumulation_steps)

        gate_alpha = self._gate_alpha_for_epoch(epoch)

        self.optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(self.train_loader, desc=f"Train {epoch}/{self.cfg.epochs}", leave=False)

        for i, (x, y) in enumerate(pbar, start=1):
            x = x.to(self.cfg.device, non_blocking=True)
            y = y.to(self.cfg.device, non_blocking=True)

            with get_autocast_ctx(self.cfg.use_amp):
                logits = self.model(x, gate_alpha=gate_alpha)
                loss = F.cross_entropy(
                    logits, y,
                    weight=self.class_weights,
                    label_smoothing=self.cfg.label_smoothing
                )

            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()

            self.scaler.scale(loss / accum_steps).backward()

            do_step = (i % accum_steps == 0) or (i == len(self.train_loader))
            if do_step:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)

                self.scaler.step(self.optimizer)
                self.scaler.update()

                self.scheduler.step()

                if self.ema is not None:
                    self.ema.update()

                self.optimizer.zero_grad(set_to_none=True)

            bs = x.size(0)
            total_loss += loss.item() * bs
            n += bs

            if i % self.cfg.log_interval == 0:
                lr = self.optimizer.param_groups[0]["lr"]
                pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{lr:.2e}", "gate_a": f"{gate_alpha:.2f}"})

        return {"loss": total_loss / max(1, n), "acc1": correct / max(1, n)}

    def _compute_gate_stats(
        self,
        gates_1d: np.ndarray,
        targets: np.ndarray,
        class_names: List[str],
        class_to_idx: Dict[str, int]
    ) -> Dict[str, Dict[str, float]]:
        out: Dict[str, Dict[str, float]] = {}
        for cname in class_names:
            ci = class_to_idx[cname]
            idx = (targets == ci)
            if idx.sum() == 0:
                out[cname] = {"mean": 0.0, "std": 0.0, "n": 0}
            else:
                vals = gates_1d[idx]
                out[cname] = {"mean": float(vals.mean()), "std": float(vals.std()), "n": int(idx.sum())}
        return out

    @torch.no_grad()
    def evaluate(
        self,
        loader: DataLoader,
        use_ema: bool = False,
        return_arrays: bool = False,
        compute_auc: bool = True
    ) -> Dict[str, object]:
        if use_ema and self.ema is not None:
            self.ema.apply_shadow()

        self.model.eval()
        all_preds, all_targets = [], []
        all_probs = [] if compute_auc else None
        all_gates = []
        total_loss, correct, n = 0.0, 0, 0

        class_names = self.train_loader.dataset.classes
        class_to_idx = self.train_loader.dataset.class_to_idx

        for x, y in tqdm(loader, desc="Eval", leave=False):
            x = x.to(self.cfg.device, non_blocking=True)
            y = y.to(self.cfg.device, non_blocking=True)

            with get_autocast_ctx(self.cfg.use_amp):
                out = self.model(x, return_gate=True, gate_alpha=1.0)
                if isinstance(out, (tuple, list)) and len(out) == 2:
                    logits, gate = out
                else:
                    logits, gate = out, None

                loss = F.cross_entropy(logits, y, weight=self.class_weights)

            probs = F.softmax(logits.float(), dim=1)
            preds = probs.argmax(dim=1)

            correct += (preds == y).sum().item()
            bs = x.size(0)
            total_loss += loss.item() * bs
            n += bs

            all_preds.extend(preds.cpu().numpy().tolist())
            all_targets.extend(y.cpu().numpy().tolist())

            if compute_auc and all_probs is not None:
                all_probs.append(probs.cpu().numpy())

            if gate is not None:
                all_gates.append(gate.detach().float().cpu().numpy())

        if use_ema and self.ema is not None:
            self.ema.restore()

        all_preds_np = np.array(all_preds)
        all_targets_np = np.array(all_targets)

        macro_f1 = f1_score(all_targets_np, all_preds_np, average="macro")
        micro_f1 = f1_score(all_targets_np, all_preds_np, average="micro")
        weighted_f1 = f1_score(all_targets_np, all_preds_np, average="weighted")
        macro_precision = precision_score(all_targets_np, all_preds_np, average="macro", zero_division=0)
        macro_recall = recall_score(all_targets_np, all_preds_np, average="macro", zero_division=0)

        macro_auc = -1.0
        all_probs_np = None
        if compute_auc and all_probs is not None:
            all_probs_np = np.concatenate(all_probs, axis=0).astype(np.float64)
            try:
                macro_auc = roc_auc_score(
                    all_targets_np,
                    all_probs_np,
                    multi_class="ovr",
                    average="macro",
                    labels=np.arange(len(class_names)),
                )
            except Exception as e:
                print(f"⚠️ AUC computation failed: {e}")
                macro_auc = -1.0

        out_dict: Dict[str, object] = {
            "loss": float(total_loss / max(1, n)),
            "acc1": float(correct / max(1, n)),
            "macro_f1": float(macro_f1),
            "micro_f1": float(micro_f1),
            "weighted_f1": float(weighted_f1),
            "macro_precision": float(macro_precision),
            "macro_recall": float(macro_recall),
            "macro_auc": float(macro_auc),
        }

        if return_arrays:
            out_dict["preds"] = all_preds_np
            out_dict["targets"] = all_targets_np
            if compute_auc and all_probs_np is not None:
                out_dict["probs"] = all_probs_np

        if self.cfg.use_hsv_branch and len(all_gates) > 0:
            gates_np = np.concatenate(all_gates, axis=0).reshape(-1)
            out_dict["gate_stats"] = self._compute_gate_stats(gates_np, all_targets_np, class_names, class_to_idx)
            out_dict["gate_global_mean"] = float(gates_np.mean())
            if return_arrays:
                out_dict["gates"] = gates_np

        return out_dict

    @torch.no_grad()
    def benchmark_inference(self, loader: DataLoader, warmup_batches: int = 5) -> Tuple[float, float]:
        self.model.eval()

        total_batches = len(loader)
        if total_batches <= warmup_batches:
            warnings.warn(
                f"Insufficient batches for benchmark: {total_batches} total, "
                f"{warmup_batches} warmup needed. Skipping throughput measurement."
            )
            return -1.0, -1.0

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = None
        n_images = 0

        for bi, (x, _y) in enumerate(loader):
            x = x.to(self.cfg.device, non_blocking=True)
            with get_autocast_ctx(self.cfg.use_amp):
                _ = self.model(x, gate_alpha=1.0)

            if bi == warmup_batches:
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                start_time = time.perf_counter()
                n_images = 0

            if start_time is not None:
                n_images += x.size(0)

        if start_time is None or n_images == 0:
            warnings.warn("Benchmark failed: no images processed post-warmup")
            return -1.0, -1.0

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - start_time
        throughput = n_images / max(elapsed, 1e-9)

        print(f"   Benchmark details: {n_images} images in {elapsed:.2f}s")
        return float(elapsed), float(throughput)

    def save_checkpoint(self, epoch: int, val_metrics: Dict[str, object], is_best: bool):
        if isinstance(self.model, nn.DataParallel):
            model_state = self.model.module.state_dict()
        else:
            model_state = self.model.state_dict()

        ckpt = {
            "epoch": int(epoch),
            "model": model_state,
            "optimizer": self.optimizer.state_dict(),
            "scheduler": self.scheduler.state_dict(),
            "scaler": self.scaler.state_dict(),
            "config": self.cfg.__dict__,
            "best_val_f1": float(self.best_val_f1),
            "bad_epochs": int(self.bad_epochs),
            "train_log": self.train_log,
        }

        if self.ema is not None:
            ckpt["ema_shadow"] = {k: v.detach().clone().cpu() for k, v in self.ema.shadow.items()}

        self.ckpt_manager.save_checkpoint(
            checkpoint=ckpt,
            epoch=epoch,
            is_best=is_best,
            save_epoch_ckpt=self.cfg.save_epoch_checkpoints,
            epoch_interval=self.cfg.save_epoch_every,
            keep_n=self.cfg.keep_last_n_checkpoints
        )

    def save_train_log(self):
        if not self.train_log:
            return
        csv_path = self.logs_dir / "train_log.csv"
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.train_log[0].keys())
            writer.writeheader()
            writer.writerows(self.train_log)

    def fit(self):
        print("\n" + "=" * 80)
        print("START TRAINING")
        print("=" * 80)
        print(f"Experiment: {self.cfg.experiment_name}")
        print(f"Model:      {self.cfg.model_name}")
        print(f"Device:     {self.cfg.device}")
        print(f"AMP:        {self.cfg.use_amp and torch.cuda.is_available()}")
        print(f"EMA:        {self.cfg.use_ema}")
        print(f"GradAccum:  {self.cfg.gradient_accumulation_steps}")
        print("=" * 80)

        effective_val_ema_used = False

        for epoch in range(self.start_epoch, self.cfg.epochs + 1):
            train_m = self.train_one_epoch(epoch)

            use_ema_for_val = self.cfg.use_ema and (self.ema is not None)
            if self.cfg.use_ema and not use_ema_for_val:
                warnings.warn("EMA requested but not available - using raw weights for validation")

            val_m = self.evaluate(
                self.val_loader,
                use_ema=use_ema_for_val,
                return_arrays=False,
                compute_auc=self.cfg.compute_val_auc
            )

            current_lr = float(self.optimizer.param_groups[0]["lr"])
            ema_tag = " (EMA)" if use_ema_for_val else ""
            print(f"\nEpoch {epoch}/{self.cfg.epochs} | LR: {current_lr:.2e}")
            print(f"  Train Loss: {train_m['loss']:.4f} | Train Acc@1: {train_m['acc1']:.4f}")
            print(
                f"  Val{ema_tag} Loss: {val_m['loss']:.4f} | "
                f"Acc@1: {val_m['acc1']:.4f} | Macro-F1: {val_m['macro_f1']:.4f} | "
                f"Micro-F1: {val_m['micro_f1']:.4f}"
            )

            if self.cfg.use_hsv_branch and "gate_stats" in val_m and "gate_global_mean" in val_m:
                self._append_gate_csv(epoch, val_m["gate_stats"], val_m["gate_global_mean"])

            self.train_log.append({
                "epoch": int(epoch),
                "lr": float(current_lr),
                "train_loss": float(train_m["loss"]),
                "train_acc1": float(train_m["acc1"]),
                "val_loss": float(val_m["loss"]),
                "val_acc1": float(val_m["acc1"]),
                "val_macro_f1": float(val_m["macro_f1"]),
                "val_micro_f1": float(val_m["micro_f1"]),
                "val_weighted_f1": float(val_m["weighted_f1"]),
                "val_macro_precision": float(val_m["macro_precision"]),
                "val_macro_recall": float(val_m["macro_recall"]),
                "val_macro_auc": float(val_m["macro_auc"]),
                "val_gate_global_mean": float(val_m.get("gate_global_mean", 0.0)),
            })
            self.save_train_log()

            improved = float(val_m["macro_f1"]) > self.best_val_f1
            if improved:
                effective_val_ema_used = use_ema_for_val
                self.best_val_f1 = float(val_m["macro_f1"])
                self.bad_epochs = 0
                self.save_checkpoint(epoch, val_m, is_best=True)
                print(f"  🌟 New best Macro-F1: {self.best_val_f1:.4f}")
            else:
                self.bad_epochs += 1
                self.save_checkpoint(epoch, val_m, is_best=False)
                print(f"  No improvement ({self.bad_epochs}/{self.cfg.early_stopping_patience})")

            if self.bad_epochs >= self.cfg.early_stopping_patience:
                print("\n⏹️ Early stopping triggered.")
                break

        print("\n" + "=" * 80)
        print("FINAL TEST EVALUATION")
        print("=" * 80)

        best_path = self.ckpt_manager.temp_dir / "best_model.pth"
        if not best_path.exists():
            raise FileNotFoundError(f"No best_model.pth found in {self.ckpt_manager.temp_dir}")

        best_ckpt = torch.load(best_path, map_location=self.cfg.device)
        target_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model

        if self.cfg.use_ema and "ema_shadow" in best_ckpt:
            print(f"📊 Loading EMA weights from best checkpoint (epoch {best_ckpt.get('epoch', 'N/A')})")
            ema_state = dict(best_ckpt["model"])
            for k, v in best_ckpt["ema_shadow"].items():
                if k in ema_state:
                    ema_state[k] = v.to(self.cfg.device, non_blocking=True)
                else:
                    warnings.warn(f"EMA shadow key '{k}' not found in model state")
            load_state_dict_robust(target_model, ema_state)
            print("✅ Loaded EMA weights for test evaluation")
            use_ema_for_test = False
            used_ema_weights = True
        elif self.cfg.use_ema:
            warnings.warn("EMA enabled but ema_shadow not found in checkpoint. Using raw weights.")
            load_state_dict_robust(target_model, best_ckpt["model"])
            use_ema_for_test = False
            used_ema_weights = False
        else:
            print(f"📊 Loading raw weights from best checkpoint (epoch {best_ckpt.get('epoch', 'N/A')})")
            load_state_dict_robust(target_model, best_ckpt["model"])
            use_ema_for_test = False
            used_ema_weights = False

        test_m = self.evaluate(
            self.test_loader,
            use_ema=use_ema_for_test,
            return_arrays=True,
            compute_auc=True
        )
        test_m["used_ema_weights"] = used_ema_weights

        infer_s, throughput = self.benchmark_inference(self.test_loader, warmup_batches=5)
        test_m["inference_seconds"] = float(infer_s)
        test_m["throughput_img_s"] = float(throughput)
        test_m["params_m"] = float(self.params_m)
        test_m["gflops"] = float(self.gflops_out)

        test_m["model_selection"] = {
            "criterion": "macro_f1",
            "cfg_use_ema": bool(self.cfg.use_ema),
            "effective_use_ema_for_validation": bool(effective_val_ema_used),
            "effective_use_ema_for_test": bool(used_ema_weights),
            "ema_decay": float(self.cfg.ema_decay) if self.cfg.use_ema else None,
            "best_epoch": int(best_ckpt.get("epoch", -1)),
            "best_val_f1": float(self.best_val_f1),
            "used_class_weights": bool(self.cfg.use_class_weights),
        }

        print(f"\n📊 Test Results:")
        print(f"   Acc@1:         {test_m['acc1']:.4f}")
        print(f"   Macro-F1:      {test_m['macro_f1']:.4f}")
        if test_m["macro_auc"] >= 0:
            print(f"   Macro-AUC:     {test_m['macro_auc']:.4f}")
        else:
            print(f"   Macro-AUC:     N/A")

        print(f"\n⚙️  Model Complexity:")
        print(f"   Params:        {test_m['params_m']:.2f}M")
        if test_m["gflops"] >= 0:
            print(f"   GFLOPs:        {test_m['gflops']:.2f}")
        else:
            print(f"   GFLOPs:        N/A")

        if test_m["inference_seconds"] >= 0:
            print(f"\n⏱️  Inference Speed (post-warmup, forward-only):")
            print(f"   Time:          {test_m['inference_seconds']:.2f}s")
            print(f"   Throughput:    {test_m['throughput_img_s']:.1f} img/s")

        self.ckpt_manager.copy_final_artifacts(
            best_ckpt_path=best_path,
            config=self.cfg,
            train_log=self.train_log,
            test_metrics=test_m,
            class_to_idx=self.train_loader.dataset.class_to_idx,
            classes=self.train_loader.dataset.classes,
            logs_dir=self.logs_dir,
        )

        return test_m


# -------------------------
# CLI
# -------------------------
def parse_args():
    p = argparse.ArgumentParser(
        description="Train Swin (RGB baseline + optional RGB+HSV gated fusion) - stabilized HSV + gate warmup",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )

    p.add_argument("--data_root", type=str, default="/kaggle/input/tea-leaf701515/tea_leaf_processed_dataset/tea_leaf_processed_dataset")
    p.add_argument("--run_dir", type=str, default="/kaggle/working/experiments_swin")
    p.add_argument("--exp_name", type=str, required=True)

    p.add_argument("--model_name", type=str, default="swin_tiny_patch4_window7_224")
    p.add_argument("--pretrained", action="store_true")
    p.add_argument("--drop_path_rate", type=float, default=0.2)

    p.add_argument("--batch_size", type=int, default=64)
    p.add_argument("--epochs", type=int, default=80)
    p.add_argument("--lr", type=float, default=5e-4)
    p.add_argument("--num_workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)

    p.add_argument("--use_ema", action="store_true")
    p.add_argument("--use_class_weights", action="store_true", help="Use inverse frequency class weights")
    p.add_argument("--gradient_accumulation_steps", type=int, default=1)
    p.add_argument("--compute_val_auc", action="store_true")

    p.add_argument("--ckpt_temp_dir", type=str, default="/kaggle/temp")
    p.add_argument("--resume", type=str, default=None)
    p.add_argument("--auto_resume", action="store_true")

    p.add_argument("--use_hsv", action="store_true")
    p.add_argument("--gate_vector", action="store_true")

    # HSV stabilization knobs
    p.add_argument("--hsv_raw", action="store_true", help="Use raw HSV (3ch) instead of sin/cos hue (4ch).")
    p.add_argument("--gate_warmup_epochs", type=int, default=5, help="Gate alpha warmup epochs (0->1).")

    p.add_argument("--save_epoch_checkpoints", action="store_true")
    p.add_argument("--save_epoch_every", type=int, default=5)
    p.add_argument("--keep_last_n_checkpoints", type=int, default=3)

    p.add_argument("--no_dp", action="store_true")
    p.add_argument("--no_cm_png", action="store_true")

    return p.parse_args()


def main():
    args = parse_args()

    cfg = Config(
        model_name=args.model_name,
        pretrained=args.pretrained,
        drop_path_rate=args.drop_path_rate,
        data_root=args.data_root,
        batch_size=args.batch_size,
        epochs=args.epochs,
        lr=args.lr,
        num_workers=args.num_workers,
        seed=args.seed,
        use_ema=args.use_ema,
        use_class_weights=args.use_class_weights,
        experiment_name=args.exp_name,
        use_hsv_branch=args.use_hsv,
        gate_vector=args.gate_vector,
        hsv_use_sincos=(not args.hsv_raw),
        gate_warmup_epochs=int(args.gate_warmup_epochs),
        ckpt_temp_dir=args.ckpt_temp_dir,
        save_epoch_checkpoints=args.save_epoch_checkpoints,
        save_epoch_every=args.save_epoch_every,
        keep_last_n_checkpoints=args.keep_last_n_checkpoints,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        compute_val_auc=args.compute_val_auc,
        run_dir=args.run_dir,
        use_data_parallel=(not args.no_dp),
        save_cm_png=(not args.no_cm_png),
    )

    print("=" * 80)
    print("ENVIRONMENT")
    print("=" * 80)
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        n_gpus = torch.cuda.device_count()
        print(f"Num GPUs: {n_gpus}")
        for i in range(n_gpus):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"PyTorch: {torch.__version__}")
    print(f"Timm:    {timm.__version__}")
    print("=" * 80)

    trainer = Trainer(cfg)

    if args.resume is not None:
        trainer.resume_from(Path(args.resume))
    elif args.auto_resume:
        last_ckpt = trainer.ckpt_manager.temp_dir / "last_checkpoint.pth"
        if last_ckpt.exists():
            trainer.resume_from(last_ckpt)
        else:
            print("ℹ️ auto_resume enabled but no last_checkpoint.pth found. Starting fresh.")

    trainer.fit()


if __name__ == "__main__":
    main()

Overwriting train_swin_three_models.py


**swin_small_hsv_vector_gate**

In [ ]:
!python train_swin_three_models.py \
  --exp_name swin_small_hsv_vector_seed42 \
  --model_name swin_small_patch4_window7_224 \
  --pretrained \
  --use_hsv \
  --gate_vector \
  --epochs 100 \
  --batch_size 64 \
  --use_ema \
  --ckpt_temp_dir /kaggle/working/ckpt_cache \
  --auto_resume \
  --num_workers 0 \
  --seed 42

In [5]:
!python train_swin_three_models.py \
  --exp_name swin_small_hsv_vector_seed1337 \
  --model_name swin_small_patch4_window7_224 \
  --pretrained \
  --use_hsv \
  --gate_vector \
  --epochs 100 \
  --batch_size 64 \
  --use_ema \
  --ckpt_temp_dir /kaggle/working/ckpt_cache \
  --auto_resume \
  --num_workers 0 \
  --seed 1337

ENVIRONMENT
CUDA Available: True
Num GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.9.0+cu126
Timm:    1.0.24

📁 Directories:
   Temp:  /kaggle/working/ckpt_cache/swin_small_hsv_vector_seed1337
   Final: /kaggle/working/experiments_swin/swin_small_hsv_vector_seed1337

🏗️  Building model: swin_small_patch4_window7_224
   Pretrained: True
   DropPath:   0.2
   RGB+HSV:    True (gate_vector=True)
   HSV sincos: True | gate warmup: 5 ep
   Params: 49.67M
   GFLOPs: 17.17

🚀 DataParallel: 2 GPUs

📦 Data: Train 6091 | Val 851 | Test 852
   Classes: ['Brown Blight', 'Gray Blight', 'Green mirid bug', 'Healthy leaf', 'Helopeltis', 'Red spider', 'Tea algal leaf spot']
   Aug note: hue jitter is DISABLED because HSV branch is enabled.

📅 Scheduler Setup (optimizer updates):
   batches/epoch: 96
   accum_steps:   1
   updates/epoch: 96
   warmup_updates:480
   total_updates: 9600

🔍 Dataset Validation:

  Class Distribution:
    Brown Blight         -> Train:   858 | Val:    93 | Test:    

In [9]:
!python train_swin_three_models.py \
  --exp_name swin_small_hsv_vector_seed2026 \
  --model_name swin_small_patch4_window7_224 \
  --pretrained \
  --use_hsv \
  --gate_vector \
  --epochs 100 \
  --batch_size 64 \
  --use_ema \
  --ckpt_temp_dir /kaggle/working/ckpt_cache \
  --auto_resume \
  --num_workers 0 \
  --seed 2026

ENVIRONMENT
CUDA Available: True
Num GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.9.0+cu126
Timm:    1.0.24

📁 Directories:
   Temp:  /kaggle/working/ckpt_cache/swin_small_hsv_vector_seed2026
   Final: /kaggle/working/experiments_swin/swin_small_hsv_vector_seed2026

🏗️  Building model: swin_small_patch4_window7_224
   Pretrained: True
   DropPath:   0.2
   RGB+HSV:    True (gate_vector=True)
   HSV sincos: True | gate warmup: 5 ep
   Params: 49.67M
   GFLOPs: 17.17

🚀 DataParallel: 2 GPUs

📦 Data: Train 6091 | Val 851 | Test 852
   Classes: ['Brown Blight', 'Gray Blight', 'Green mirid bug', 'Healthy leaf', 'Helopeltis', 'Red spider', 'Tea algal leaf spot']
   Aug note: hue jitter is DISABLED because HSV branch is enabled.

📅 Scheduler Setup (optimizer updates):
   batches/epoch: 96
   accum_steps:   1
   updates/epoch: 96
   warmup_updates:480
   total_updates: 9600

🔍 Dataset Validation:

  Class Distribution:
    Brown Blight         -> Train:   858 | Val:    93 | Test:    

**Report**

In [18]:
import os, json, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = "/kaggle/working/experiments_swin"
OUT_DIR = os.path.join(ROOT, "_summary")
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# TARGET RUNS (ONLY THESE)
# ----------------------------
TARGET_RUNS = {
    "swin_small_hsv_vector_seed1337",
    "swin_small_hsv_vector_seed2026",
    "swin_small_hsv_vector_seed42",
}

# ----------------------------
# Helpers
# ----------------------------
def infer_seed(run_name: str):
    m = re.search(r"seed(\d+)", run_name)
    return int(m.group(1)) if m else None

def safe_read_json(path):
    with open(path, "r") as f:
        return json.load(f)

def format_mean_std(mean, std, digits=4):
    if pd.isna(mean) or pd.isna(std):
        return ""
    return f"{mean:.{digits}f} ± {std:.{digits}f}"

def save_df(df, name):
    csv_path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(csv_path, index=False)
    return csv_path

def normalize_confmat(cm):
    cm = cm.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return cm / row_sums

def plot_confmat(cm, labels, title, outpath, normalize_rows=True):
    cm_plot = normalize_confmat(cm) if normalize_rows else cm

    plt.figure(figsize=(10, 8))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45, ha="right")
    plt.yticks(ticks, labels)
    plt.tight_layout()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.savefig(outpath, dpi=1200, bbox_inches="tight")
    plt.close()

# ----------------------------
# Discover runs (ONLY targets)
# ----------------------------
run_dirs_all = sorted([d for d in glob.glob(os.path.join(ROOT, "*")) if os.path.isdir(d)])
run_dirs = [d for d in run_dirs_all if os.path.basename(d) in TARGET_RUNS]

missing = sorted(list(TARGET_RUNS - set(map(os.path.basename, run_dirs))))
if missing:
    print("WARNING: These target runs were not found under ROOT:")
    for m in missing:
        print(" -", m)

rows = []
per_class_records = []
confmats = []

for run_dir in run_dirs:
    run_name = os.path.basename(run_dir)
    metrics_dir = os.path.join(run_dir, "metrics")
    if not os.path.isdir(metrics_dir):
        continue

    test_json = os.path.join(metrics_dir, "test_results.json")
    per_class_json = os.path.join(metrics_dir, "per_class_metrics.json")
    conf_npy = os.path.join(metrics_dir, "confusion_matrix.npy")

    if not os.path.exists(test_json):
        continue

    seed = infer_seed(run_name)

    tr = safe_read_json(test_json)

    # FORCE EMA = True for all three runs (your request)
    used_ema_forced = True

    row = {
        "run": run_name,
        "used_ema": used_ema_forced,
        "seed": seed,
        "loss": tr.get("loss", np.nan),
        "acc1": tr.get("acc1", np.nan),
        "macro_f1": tr.get("macro_f1", np.nan),
        "micro_f1": tr.get("micro_f1", np.nan),
        "weighted_f1": tr.get("weighted_f1", np.nan),
        "macro_precision": tr.get("macro_precision", np.nan),
        "macro_recall": tr.get("macro_recall", np.nan),
        "macro_auc": tr.get("macro_auc", np.nan),
        "inference_seconds": tr.get("inference_seconds", np.nan),
        "throughput_img_s": tr.get("throughput_img_s", np.nan),
        "params_m": tr.get("params_m", np.nan),
        "gflops": tr.get("gflops", np.nan),
    }
    rows.append(row)

    # Per-class metrics
    if os.path.exists(per_class_json):
        pcm = safe_read_json(per_class_json)
        for cls, m in pcm.items():
            per_class_records.append({
                "run": run_name,
                "used_ema": used_ema_forced,
                "seed": seed,
                "class": cls,
                "precision": m.get("precision", np.nan),
                "recall": m.get("recall", np.nan),
                "f1": m.get("f1", np.nan),
            })

    # Confusion matrix
    if os.path.exists(conf_npy):
        cm = np.load(conf_npy)
        confmats.append((run_name, used_ema_forced, seed, cm))

df_runs = pd.DataFrame(rows)
df_pc = pd.DataFrame(per_class_records)

if df_runs.empty:
    raise RuntimeError(f"No target runs found with test_results.json under: {ROOT}")

# ----------------------------
# Per-run table + best run
# ----------------------------
df_runs_sorted = df_runs.sort_values(["seed", "run"]).reset_index(drop=True)

df_best = df_runs_sorted.sort_values(["acc1", "macro_f1"], ascending=False).head(1)

save_df(df_runs_sorted, "runs_all")
save_df(df_best, "best_run")

# ----------------------------
# Group summary: ONLY EMA (mean ± std across seeds)
# ----------------------------
group_cols = ["loss", "acc1", "macro_f1", "micro_f1", "weighted_f1",
              "macro_precision", "macro_recall", "macro_auc",
              "inference_seconds", "throughput_img_s", "params_m", "gflops"]

stats = df_runs_sorted[group_cols].agg(["mean", "std", "min", "max", "count"]).T.reset_index()
stats.columns = ["metric", "mean", "std", "min", "max", "count"]

# nice one-row table like before
nice = {"condition": "EMA", "n_runs": int(df_runs_sorted.shape[0])}
for c in group_cols:
    nice[c] = format_mean_std(df_runs_sorted[c].mean(), df_runs_sorted[c].std(), digits=4)
df_group_nice = pd.DataFrame([nice])

save_df(df_group_nice, "group_summary_mean_std")

# ----------------------------
# Per-class summary (mean ± std across seeds), EMA only
# ----------------------------
if not df_pc.empty:
    pc_stats = (
        df_pc
        .groupby(["class"])
        .agg(
            precision_mean=("precision", "mean"),
            precision_std=("precision", "std"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", "std"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            n=("f1", "count"),
        )
        .reset_index()
    )

    pc_stats["precision_meanstd"] = pc_stats.apply(lambda r: format_mean_std(r["precision_mean"], r["precision_std"], 4), axis=1)
    pc_stats["recall_meanstd"]    = pc_stats.apply(lambda r: format_mean_std(r["recall_mean"], r["recall_std"], 4), axis=1)
    pc_stats["f1_meanstd"]        = pc_stats.apply(lambda r: format_mean_std(r["f1_mean"], r["f1_std"], 4), axis=1)

    out_f1 = pc_stats[["class", "f1_meanstd"]].rename(columns={"f1_meanstd": "EMA"})
    out_p  = pc_stats[["class", "precision_meanstd"]].rename(columns={"precision_meanstd": "EMA"})
    out_r  = pc_stats[["class", "recall_meanstd"]].rename(columns={"recall_meanstd": "EMA"})

    out_f1.to_csv(os.path.join(OUT_DIR, "per_class_f1_mean_std.csv"), index=False)
    out_p.to_csv(os.path.join(OUT_DIR, "per_class_precision_mean_std.csv"), index=False)
    out_r.to_csv(os.path.join(OUT_DIR, "per_class_recall_mean_std.csv"), index=False)

# ----------------------------
# Confusion matrices: summed and normalized plots (All = EMA here)
# ----------------------------
labels = None
if not df_pc.empty:
    labels = sorted(df_pc["class"].unique().tolist())

all_cms = [cm for (_, _, _, cm) in confmats]

def sum_cms(cms):
    if not cms:
        return None
    s = np.zeros_like(cms[0], dtype=np.int64)
    for cm in cms:
        s += cm.astype(np.int64)
    return s

cm_all = sum_cms(all_cms)

if cm_all is not None:
    np.save(os.path.join(OUT_DIR, "confmat_sum_all.npy"), cm_all)

if cm_all is not None and labels is None:
    labels = [f"C{i}" for i in range(cm_all.shape[0])]

if cm_all is not None:
    plot_confmat(
        cm_all, labels,
        "Confusion Matrix (Summed, Selected Runs) - Row Normalized",
        os.path.join(OUT_DIR, "confmat_all_normalized.png"),
        normalize_rows=True
    )

# ----------------------------
# Console report
# ----------------------------
def df_to_markdown(df, max_rows=30):
    try:
        return df.head(max_rows).to_markdown(index=False)
    except Exception:
        return df.head(max_rows).to_string(index=False)

print("\n" + "="*80)
print("FULL PROFESSIONAL RESULTS SUMMARY (Selected HSV Vector Runs, EMA forced)")
print("="*80)

print("\n[1] Per-run results (key metrics):")
key_cols = ["run", "used_ema", "seed", "acc1", "macro_f1", "weighted_f1",
            "macro_precision", "macro_recall", "macro_auc", "loss"]
print(df_to_markdown(df_runs_sorted[key_cols].sort_values(["seed"])))

print("\n[2] Group summary: EMA only (mean ± std across seeds):")
display_cols = ["condition", "n_runs", "acc1", "macro_f1", "weighted_f1",
                "macro_precision", "macro_recall", "macro_auc", "loss"]
print(df_to_markdown(df_group_nice[display_cols]))

print("\n[3] Best run (by acc1 then macro_f1):")
print(df_to_markdown(df_best[key_cols]))

if not df_pc.empty:
    print("\n[4] Per-class F1 (mean ± std) (EMA only):")
    wide_f1 = pd.read_csv(os.path.join(OUT_DIR, "per_class_f1_mean_std.csv"))
    print(df_to_markdown(wide_f1.sort_values("class")))

print("\n[5] Saved artifacts in:", OUT_DIR)
print(" - runs_all.csv")
print(" - best_run.csv")
print(" - group_summary_mean_std.csv")
if not df_pc.empty:
    print(" - per_class_f1_mean_std.csv")
    print(" - per_class_precision_mean_std.csv")
    print(" - per_class_recall_mean_std.csv")
if cm_all is not None:
    print(" - confmat_all_normalized.png + confmat_sum_all.npy")
print("="*80)

summary = {
    "root": ROOT,
    "out_dir": OUT_DIR,
    "target_runs": sorted(list(TARGET_RUNS)),
    "n_runs": int(len(df_runs_sorted)),
    "best_run": df_best.to_dict(orient="records")[0],
    "group_summary_mean_std": df_group_nice.to_dict(orient="records"),
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote:", os.path.join(OUT_DIR, "summary.json"))


FULL PROFESSIONAL RESULTS SUMMARY (Selected HSV Vector Runs, EMA forced)

[1] Per-run results (key metrics):
| run                            | used_ema   |   seed |     acc1 |   macro_f1 |   weighted_f1 |   macro_precision |   macro_recall |   macro_auc |     loss |
|:-------------------------------|:-----------|-------:|---------:|-----------:|--------------:|------------------:|---------------:|------------:|---------:|
| swin_small_hsv_vector_seed42   | True       |     42 | 0.954225 |   0.948531 |      0.954208 |          0.946928 |       0.950845 |    0.995654 | 0.281391 |
| swin_small_hsv_vector_seed1337 | True       |   1337 | 0.963615 |   0.959223 |      0.963658 |          0.957975 |       0.960722 |    0.996063 | 0.30382  |
| swin_small_hsv_vector_seed2026 | True       |   2026 | 0.953052 |   0.949905 |      0.952916 |          0.949089 |       0.951835 |    0.996431 | 0.282962 |

[2] Group summary: EMA only (mean ± std across seeds):
| condition   |   n_runs | acc1        

In [5]:
%%writefile robustness_eval_swin_hsv_vector.py
import os
import io
import math
import json
import argparse
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import warnings
warnings.filterwarnings("ignore", message=".*pin_memory.*")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torchvision.datasets import ImageFolder

import timm
from tqdm import tqdm

from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# -------------------------
# Repro / utils
# -------------------------
def set_seed(seed: int = 42, deterministic: bool = True):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass


def strip_module_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            out[k[7:]] = v
        else:
            out[k] = v
    return out


def add_module_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in state_dict.items():
        out[k if k.startswith("module.") else f"module.{k}"] = v
    return out


def strip_prefix(state_dict: Dict[str, torch.Tensor], prefix: str) -> Dict[str, torch.Tensor]:
    out = {}
    for k, v in state_dict.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
        else:
            out[k] = v
    return out


def load_state_dict_robust(model: nn.Module, state_dict: Dict[str, torch.Tensor], verbose: bool = True) -> None:
    """
    Robust load for common save formats:
    - plain keys
    - module.* keys
    - backbone.* keys
    """
    candidates = [
        state_dict,
        strip_module_prefix(state_dict),
        add_module_prefix(state_dict),
        strip_prefix(state_dict, "backbone."),
        strip_module_prefix(strip_prefix(state_dict, "backbone.")),
        add_module_prefix(strip_prefix(state_dict, "backbone.")),
    ]

    for sd in candidates:
        try:
            model.load_state_dict(sd, strict=True)
            if verbose:
                print("[OK] Loaded weights with strict=True")
            return
        except Exception:
            pass

    for sd in candidates:
        try:
            res = model.load_state_dict(sd, strict=False)
            if verbose:
                mk = getattr(res, "missing_keys", [])
                uk = getattr(res, "unexpected_keys", [])
                if mk:
                    print(f"[WARN] Missing keys ({len(mk)}): e.g. {mk[:6]}")
                if uk:
                    print(f"[WARN] Unexpected keys ({len(uk)}): e.g. {uk[:6]}")
                print("[OK] Loaded weights with strict=False")
            return
        except Exception:
            pass

    model_keys = set(model.state_dict().keys())
    filtered = {k: v for k, v in state_dict.items() if k in model_keys}
    if filtered:
        res = model.load_state_dict(filtered, strict=False)
        if verbose:
            mk = getattr(res, "missing_keys", [])
            uk = getattr(res, "unexpected_keys", [])
            if mk:
                print(f"[WARN] Missing keys ({len(mk)}): e.g. {mk[:6]}")
            if uk:
                print(f"[WARN] Unexpected keys ({len(uk)}): e.g. {uk[:6]}")
            print("[OK] Loaded filtered weights with strict=False")
        return

    raise RuntimeError("Could not load state dict with any strategy.")


# -------------------------
# HSV helpers (same as training)
# -------------------------
def rgb_to_hsv_torch(rgb01: torch.Tensor) -> torch.Tensor:
    r, g, b = rgb01[:, 0:1], rgb01[:, 1:2], rgb01[:, 2:3]
    maxc, _ = rgb01.max(dim=1, keepdim=True)
    minc, _ = rgb01.min(dim=1, keepdim=True)
    v = maxc
    delta = maxc - minc

    eps = 1e-10
    s = delta / torch.clamp(maxc, min=eps)
    s = torch.where(maxc > eps, s, torch.zeros_like(s))

    delta_safe = torch.clamp(delta, min=eps)

    h_r = (g - b) / delta_safe
    h_g = (b - r) / delta_safe + 2.0
    h_b = (r - g) / delta_safe + 4.0

    idx = rgb01.argmax(dim=1, keepdim=True)
    mask = (delta > eps)

    h = torch.zeros_like(delta, dtype=rgb01.dtype)
    h = torch.where((idx == 0) & mask, h_r, h)
    h = torch.where((idx == 1) & mask, h_g, h)
    h = torch.where((idx == 2) & mask, h_b, h)

    h = torch.remainder(h / 6.0, 1.0)
    hsv = torch.cat([h, s, v], dim=1)
    return torch.clamp(hsv, 0.0, 1.0)


def hsv_to_sincos_sv(hsv01: torch.Tensor) -> torch.Tensor:
    h = hsv01[:, 0:1]
    s = hsv01[:, 1:2]
    v = hsv01[:, 2:3]
    ang = 2.0 * math.pi * h
    hsin = torch.sin(ang)
    hcos = torch.cos(ang)
    return torch.cat([hsin, hcos, s, v], dim=1)


def normalize_hsv_rep(x: torch.Tensor) -> torch.Tensor:
    hsin = x[:, 0:1]
    hcos = x[:, 1:2]
    s = (x[:, 2:3] - 0.5) / 0.25
    v = (x[:, 3:4] - 0.5) / 0.25
    return torch.cat([hsin, hcos, s, v], dim=1)


# -------------------------
# Model (HSV VECTOR gate)
# -------------------------
class SwinRGBHSV(nn.Module):
    def __init__(
        self,
        model_name: str,
        pretrained: bool,
        drop_path_rate: float,
        num_classes: int,
        use_hsv_branch: bool,
        hsv_embed_dim: int = 128,
        hsv_use_sincos: bool = True,
        hsv_dropout: float = 0.1,
        fuse_dropout: float = 0.2,
        gate_hidden: int = 256,
        gate_vector: bool = True,
        img_mean: Tuple[float, float, float] = (0.485, 0.456, 0.406),
        img_std: Tuple[float, float, float] = (0.229, 0.224, 0.225),
    ):
        super().__init__()
        self.use_hsv_branch = use_hsv_branch
        self.hsv_use_sincos = hsv_use_sincos
        self.gate_vector = gate_vector

        self.register_buffer("img_mean", torch.tensor(img_mean).view(1, 3, 1, 1))
        self.register_buffer("img_std", torch.tensor(img_std).view(1, 3, 1, 1))

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            drop_path_rate=drop_path_rate,
            global_pool="avg",
        )
        feat_dim = int(getattr(self.backbone, "num_features", 768))
        self.feat_dim = feat_dim

        if self.use_hsv_branch:
            in_ch = 4 if self.hsv_use_sincos else 3

            self.hsv_branch = nn.Sequential(
                nn.Conv2d(in_ch, 16, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(16),
                nn.GELU(),
                nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.GELU(),
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(p=hsv_dropout),
                nn.Linear(32, hsv_embed_dim),
                nn.GELU(),
            )

            self.hsv_proj = nn.Sequential(
                nn.Dropout(p=fuse_dropout),
                nn.Linear(hsv_embed_dim, self.feat_dim),
            )

            gate_out_dim = self.feat_dim if self.gate_vector else 1
            self.gate_mlp = nn.Sequential(
                nn.Linear(self.feat_dim + hsv_embed_dim, gate_hidden),
                nn.GELU(),
                nn.Dropout(p=fuse_dropout),
                nn.Linear(gate_hidden, gate_out_dim),
            )
            if self.gate_mlp[-1].bias is not None:
                nn.init.constant_(self.gate_mlp[-1].bias, -2.0)

            self.classifier = nn.Sequential(
                nn.Dropout(p=fuse_dropout),
                nn.Linear(self.feat_dim, self.feat_dim // 2),
                nn.GELU(),
                nn.Dropout(p=fuse_dropout),
                nn.Linear(self.feat_dim // 2, num_classes),
            )
        else:
            self.classifier = nn.Linear(self.feat_dim, num_classes)

    def forward(self, x_norm: torch.Tensor, return_gate: bool = False, gate_alpha: float = 1.0):
        feat = self.backbone(x_norm)

        if not self.use_hsv_branch:
            logits = self.classifier(feat)
            if return_gate:
                gate = torch.zeros((logits.size(0), 1), device=logits.device, dtype=logits.dtype)
                return logits, gate
            return logits

        # FP32 HSV conversion (AMP-safe)
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast(enabled=False):
                x_rgb01 = (x_norm.float() * self.img_std.float()) + self.img_mean.float()
                x_rgb01 = torch.clamp(x_rgb01, 0.0, 1.0)
                x_hsv = rgb_to_hsv_torch(x_rgb01)
                if self.hsv_use_sincos:
                    x_hsv_rep = normalize_hsv_rep(hsv_to_sincos_sv(x_hsv))
                else:
                    x_hsv_rep = x_hsv
        else:
            x_rgb01 = (x_norm.float() * self.img_std.float()) + self.img_mean.float()
            x_rgb01 = torch.clamp(x_rgb01, 0.0, 1.0)
            x_hsv = rgb_to_hsv_torch(x_rgb01)
            x_hsv_rep = normalize_hsv_rep(hsv_to_sincos_sv(x_hsv)) if self.hsv_use_sincos else x_hsv

        x_hsv_rep = x_hsv_rep.to(dtype=feat.dtype)
        hsv_emb = self.hsv_branch(x_hsv_rep)

        gate_in = torch.cat([feat.float(), hsv_emb.float()], dim=1)
        gate = torch.sigmoid(self.gate_mlp(gate_in))

        if not self.gate_vector:
            gate_for_fuse = gate.expand(-1, self.feat_dim)
            gate_stat = gate
        else:
            gate_for_fuse = gate
            gate_stat = gate.mean(dim=1, keepdim=True)

        hsv_feat = self.hsv_proj(hsv_emb).to(dtype=feat.dtype)
        gate_for_fuse = gate_for_fuse.to(dtype=feat.dtype)

        fused = feat + (float(gate_alpha) * gate_for_fuse) * hsv_feat
        logits = self.classifier(fused)

        if return_gate:
            return logits, gate_stat
        return logits


# -------------------------
# Corruptions (on x01 tensor, then re-normalize)
# -------------------------
def clamp01(x: torch.Tensor) -> torch.Tensor:
    return torch.clamp(x, 0.0, 1.0)


def apply_brightness(x01: torch.Tensor, delta: float) -> torch.Tensor:
    return clamp01(x01 + delta)


def apply_contrast(x01: torch.Tensor, factor: float) -> torch.Tensor:
    mean = x01.mean(dim=(2, 3), keepdim=True)
    return clamp01((x01 - mean) * factor + mean)


def apply_gaussian_noise(x01: torch.Tensor, sigma: float) -> torch.Tensor:
    if sigma <= 0:
        return x01
    return clamp01(x01 + torch.randn_like(x01) * sigma)


def apply_color_temperature(x01: torch.Tensor, strength: float) -> torch.Tensor:
    s = float(np.clip(strength, -1.0, 1.0))
    r_gain = 1.0 + 0.20 * s
    b_gain = 1.0 - 0.20 * s
    g_gain = 1.0 + 0.05 * s
    out = x01.clone()
    out[:, 0:1] *= r_gain
    out[:, 1:2] *= g_gain
    out[:, 2:3] *= b_gain
    return clamp01(out)


def apply_hue_shift(x01: torch.Tensor, hue_delta: float) -> torch.Tensor:
    hsv = rgb_to_hsv_torch(x01)
    h = torch.remainder(hsv[:, 0:1] + hue_delta, 1.0)
    s = hsv[:, 1:2]
    v = hsv[:, 2:3]

    h6 = h * 6.0
    i = torch.floor(h6).to(torch.int64)
    f = h6 - i.float()
    p = v * (1.0 - s)
    q = v * (1.0 - s * f)
    t = v * (1.0 - s * (1.0 - f))

    i_mod = torch.remainder(i, 6)
    r = torch.zeros_like(v); g = torch.zeros_like(v); b = torch.zeros_like(v)

    cond = (i_mod == 0)
    r = torch.where(cond, v, r); g = torch.where(cond, t, g); b = torch.where(cond, p, b)
    cond = (i_mod == 1)
    r = torch.where(cond, q, r); g = torch.where(cond, v, g); b = torch.where(cond, p, b)
    cond = (i_mod == 2)
    r = torch.where(cond, p, r); g = torch.where(cond, v, g); b = torch.where(cond, t, b)
    cond = (i_mod == 3)
    r = torch.where(cond, p, r); g = torch.where(cond, q, g); b = torch.where(cond, v, b)
    cond = (i_mod == 4)
    r = torch.where(cond, t, r); g = torch.where(cond, p, g); b = torch.where(cond, v, b)
    cond = (i_mod == 5)
    r = torch.where(cond, v, r); g = torch.where(cond, p, g); b = torch.where(cond, q, b)

    return clamp01(torch.cat([r, g, b], dim=1))


def apply_jpeg_compression_batch(x01_bchw: torch.Tensor, quality: int) -> torch.Tensor:
    """
    JPEG compression simulation (PIL). Runs on CPU per-image.
    Accepts BCHW float in [0,1] and returns BCHW float in [0,1].
    """
    q = int(np.clip(quality, 1, 100))
    outs = []
    x_cpu = (x01_bchw.detach().clamp(0, 1) * 255.0).round().to(torch.uint8).cpu()

    for i in range(x_cpu.size(0)):
        arr = x_cpu[i].permute(1, 2, 0).numpy()  # HWC uint8
        img = Image.fromarray(arr)
        if img.mode != "RGB":
            img = img.convert("RGB")

        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q, optimize=True)
        buf.seek(0)

        with Image.open(buf) as im:
            im = im.convert("RGB")
            arr_j = np.array(im, dtype=np.uint8)

        out = torch.from_numpy(arr_j).permute(2, 0, 1).contiguous().float() / 255.0
        outs.append(out)

    out_bchw = torch.stack(outs, dim=0).to(device=x01_bchw.device, dtype=x01_bchw.dtype)
    return out_bchw


def make_conditions() -> List[Dict]:
    conds: List[Dict] = []
    conds.append({"name": "clean", "ops": []})

    for pct in [10, 20, 30, 40]:
        d = pct / 100.0
        conds.append({"name": f"brightness_+{pct}%", "ops": [("brightness", +d)]})
        conds.append({"name": f"brightness_-{pct}%", "ops": [("brightness", -d)]})

    for c in [0.6, 0.8, 1.2, 1.4]:
        conds.append({"name": f"contrast_x{c:.1f}", "ops": [("contrast", c)]})

    for s in [-1.0, -0.5, 0.5, 1.0]:
        tag = "cool" if s < 0 else "warm"
        conds.append({"name": f"temp_{tag}_{abs(s):.1f}", "ops": [("temp", s)]})

    for h in [0.15, 0.25]:
        conds.append({"name": f"hue_+{h:.2f}", "ops": [("hue", +h)]})
        conds.append({"name": f"hue_-{h:.2f}", "ops": [("hue", -h)]})

    for s in [0.01, 0.03, 0.05, 0.08]:
        conds.append({"name": f"gauss_sigma_{s:.2f}", "ops": [("gauss", s)]})

    # JPEG qualities (app/mobile)
    for q in [95, 85, 75, 60, 45, 30]:
        conds.append({"name": f"jpeg_q{q}", "ops": [("jpeg", q)]})

    # app-like combos
    conds.append({"name": "app_like_q75_bright+10", "ops": [("brightness", +0.10), ("jpeg", 75)]})
    conds.append({"name": "app_like_q60_contrast1.2", "ops": [("contrast", 1.2), ("jpeg", 60)]})
    conds.append({"name": "app_like_q45_lowlight", "ops": [("brightness", -0.20), ("gauss", 0.03), ("jpeg", 45)]})

    # composites
    conds.append({"name": "domain_field_sunlight", "ops": [("brightness", +0.15), ("contrast", 1.3), ("temp", +0.5)]})
    conds.append({"name": "domain_cloudy_shadow", "ops": [("brightness", -0.15), ("contrast", 0.8), ("temp", -0.5)]})
    conds.append({"name": "domain_lowlight_phone", "ops": [("brightness", -0.25), ("contrast", 0.9), ("gauss", 0.05), ("temp", -0.2)]})
    conds.append({"name": "domain_lowlight_phone_jpeg_q45", "ops": [("brightness", -0.25), ("contrast", 0.9), ("gauss", 0.05), ("temp", -0.2), ("jpeg", 45)]})

    return conds


def apply_ops_batch(x01: torch.Tensor, ops: List[Tuple[str, float]]) -> torch.Tensor:
    out = x01
    for (op, val) in ops:
        if op == "brightness":
            out = apply_brightness(out, float(val))
        elif op == "contrast":
            out = apply_contrast(out, float(val))
        elif op == "gauss":
            out = apply_gaussian_noise(out, float(val))
        elif op == "temp":
            out = apply_color_temperature(out, float(val))
        elif op == "hue":
            out = apply_hue_shift(out, float(val))
        elif op == "jpeg":
            out = apply_jpeg_compression_batch(out, int(val))
        else:
            raise ValueError(f"Unknown op: {op}")
    return out


# -------------------------
# Evaluation
# -------------------------
@torch.no_grad()
def eval_one_condition(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    img_mean: Tuple[float, float, float],
    img_std: Tuple[float, float, float],
    ops: List[Tuple[str, float]],
    use_amp: bool = True,
    return_gate: bool = True,
) -> Dict[str, float]:
    model.eval()
    mean_t = torch.tensor(img_mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(img_std, device=device).view(1, 3, 1, 1)

    y_true, y_pred = [], []
    gate_vals = []

    autocast_ctx = (
        torch.amp.autocast(device_type="cuda", enabled=(use_amp and torch.cuda.is_available()))
        if hasattr(torch, "amp") and hasattr(torch.amp, "autocast")
        else torch.cuda.amp.autocast(enabled=(use_amp and torch.cuda.is_available()))
    )

    for x_norm, y in tqdm(loader, desc="Eval", leave=False):
        x_norm = x_norm.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        x01 = torch.clamp(x_norm * std_t + mean_t, 0.0, 1.0)
        x01c = apply_ops_batch(x01, ops)
        x_in = (x01c - mean_t) / std_t

        with autocast_ctx:
            out = model(x_in, return_gate=return_gate, gate_alpha=1.0)
            if isinstance(out, (tuple, list)) and len(out) == 2:
                logits, gate = out
            else:
                logits, gate = out, None

        preds = logits.argmax(dim=1)
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())

        if gate is not None:
            gate_vals.append(gate.detach().float().cpu().numpy())

    y_true_np = np.array(y_true)
    y_pred_np = np.array(y_pred)

    outm = {
        "acc": float(accuracy_score(y_true_np, y_pred_np)),
        "macro_f1": float(f1_score(y_true_np, y_pred_np, average="macro")),
        "macro_precision": float(precision_score(y_true_np, y_pred_np, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true_np, y_pred_np, average="macro", zero_division=0)),
    }

    if gate_vals:
        g = np.concatenate(gate_vals, axis=0).reshape(-1)
        outm["gate_mean"] = float(g.mean())
        outm["gate_std"] = float(g.std())
    else:
        outm["gate_mean"] = float("nan")
        outm["gate_std"] = float("nan")

    return outm


def save_csv(rows: List[Dict], out_csv: str):
    import csv
    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(rows[0].keys()) if rows else []
    with open(out_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print(f"[OK] Wrote: {out_path}")


def main():
    p = argparse.ArgumentParser("Robustness eval for Swin RGB+HSV (vector gate) + JPEG")
    p.add_argument("--test_dir", type=str, required=True, help=".../test")
    p.add_argument("--weights", type=str, required=True, help="Path to ema_model_weights_only.pth or model_weights_only.pth")
    p.add_argument("--model_name", type=str, default="swin_small_patch4_window7_224")
    p.add_argument("--drop_path_rate", type=float, default=0.2)
    p.add_argument("--batch_size", type=int, default=64)
    p.add_argument("--num_workers", type=int, default=0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--no_amp", action="store_true")
    p.add_argument("--out_csv", type=str, default="/kaggle/working/robustness_swin_hsv_vector.csv")
    args = p.parse_args()

    set_seed(args.seed, deterministic=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Device: {device}")

    img_mean = (0.485, 0.456, 0.406)
    img_std = (0.229, 0.224, 0.225)

    eval_tf = T.Compose([
        T.Resize(int(224 * 1.14), interpolation=InterpolationMode.BICUBIC),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=img_mean, std=img_std),
    ])

    test_ds = ImageFolder(args.test_dir, transform=eval_tf)
    test_loader = DataLoader(
        test_ds,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=(device == "cuda"),
        persistent_workers=(args.num_workers > 0),
    )
    print(f"[INFO] Test images: {len(test_ds)} | classes: {len(test_ds.classes)}")
    print(f"[INFO] Classes: {test_ds.classes}")

    # HSV VECTOR model: --use_hsv --gate_vector and (no --hsv_raw) => hsv_use_sincos=True
    model = SwinRGBHSV(
        model_name=args.model_name,
        pretrained=False,
        drop_path_rate=args.drop_path_rate,
        num_classes=len(test_ds.classes),
        use_hsv_branch=True,
        hsv_use_sincos=True,
        gate_vector=True,
        img_mean=img_mean,
        img_std=img_std,
    ).to(device)

    ckpt = torch.load(args.weights, map_location="cpu")
    if isinstance(ckpt, dict) and "model" in ckpt and isinstance(ckpt["model"], dict):
        state = ckpt["model"]
    else:
        state = ckpt if isinstance(ckpt, dict) else ckpt

    load_state_dict_robust(model, state, verbose=True)
    print(f"[OK] Loaded weights from: {args.weights}")

    rows: List[Dict] = []
    for c in make_conditions():
        name = c["name"]
        ops = c["ops"]
        print("\n" + "=" * 80)
        print(f"[RUN] Condition: {name}")
        print(f"      Ops: {ops}")
        print("=" * 80)

        metrics = eval_one_condition(
            model=model,
            loader=test_loader,
            device=device,
            img_mean=img_mean,
            img_std=img_std,
            ops=ops,
            use_amp=(not args.no_amp),
            return_gate=True,
        )
        row = {"condition": name, **metrics}
        print(f"[RES] {row}")
        rows.append(row)

    save_csv(rows, args.out_csv)

    out_json = str(Path(args.out_csv).with_suffix(".json"))
    with open(out_json, "w") as f:
        json.dump(rows, f, indent=2)
    print(f"[OK] Wrote: {out_json}")


if __name__ == "__main__":
    main()

Overwriting robustness_eval_swin_hsv_vector.py


In [6]:
!python robustness_eval_swin_hsv_vector.py \
  --test_dir "/kaggle/input/tea-leaf701515/tea_leaf_processed_dataset/tea_leaf_processed_dataset/test" \
  --weights "/kaggle/working/experiments_swin/swin_small_hsv_vector_seed1337/model/ema_model_weights_only.pth" \
  --model_name "swin_small_patch4_window7_224" \
  --batch_size 64 \
  --num_workers 0 \
  --out_csv "/kaggle/working/experiments_swin/swin_small_hsv_vector_seed1337/metrics/robustness_swin_small_hsv_vector_ema_seed1337.csv"

[INFO] Device: cpu
[INFO] Test images: 852 | classes: 7
[INFO] Classes: ['Brown Blight', 'Gray Blight', 'Green mirid bug', 'Healthy leaf', 'Helopeltis', 'Red spider', 'Tea algal leaf spot']
[OK] Loaded weights with strict=True
[OK] Loaded weights from: /kaggle/working/experiments_swin/swin_small_hsv_vector_seed1337/model/ema_model_weights_only.pth

[RUN] Condition: clean
      Ops: []
[RES] {'condition': 'clean', 'acc': 0.9636150234741784, 'macro_f1': 0.9592232927095996, 'macro_precision': 0.9579746675596129, 'macro_recall': 0.9607219869873898, 'gate_mean': 0.10016807913780212, 'gate_std': 0.023621106520295143}

[RUN] Condition: brightness_+10%
      Ops: [('brightness', 0.1)]
[RES] {'condition': 'brightness_+10%', 'acc': 0.9647887323943662, 'macro_f1': 0.9600549130843431, 'macro_precision': 0.9587095532572955, 'macro_recall': 0.961711795412849, 'gate_mean': 0.1002298966050148, 'gate_std': 0.023512231186032295}

[RUN] Condition: brightness_-10%
      Ops: [('brightness', -0.1)]
[RES] {

In [1]:
%%writefile generate_pseudo_masks.py
import os
import sys
import math
import random
import warnings
import argparse
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image
import cv2
from tqdm import tqdm

warnings.filterwarnings("ignore")

try:
    from train_swin_three_models import SwinRGBHSV, rgb_to_hsv_torch, hsv_to_sincos_sv, normalize_hsv_rep
except ImportError:
    raise ImportError(
        "Could not import from train_swin_three_models.py.\n"
        "Make sure generate_pseudo_masks.py is in the same directory as "
        "train_swin_three_models.py, or paste the required classes above."
    )

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  — edit these paths before running
# ─────────────────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = "/kaggle/working/experiments_swin/swin_small_hsv_vector_seed1337/model/ema_model_weights_only.pth"

DATA_ROOT    = "/kaggle/input/tea-leaf701515/tea_leaf_processed_dataset/tea_leaf_processed_dataset"
MASK_OUT_DIR = "/kaggle/working/pseudo_masks/train"

IMG_SIZE         = 224
THRESHOLD_PCT    = 70     # GradCAM percentile threshold for binarization
MORPH_KERNEL     = 5      # morphological cleanup kernel size (set 0 to skip)
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE       = 1      # GradCAM must run per-image (no batch support)
SKIP_EXISTING    = True   # Resume-friendly: skip already-generated masks


class GradCAM:
    """
    Minimal GradCAM for Swin backbone.
    Hooks into the target layer, runs a forward+backward pass,
    and returns a (H, W) heatmap in [0, 1].
    """
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            # Swin norm1 output shape: (B, N, C) where N = num_tokens
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def __call__(self, x: torch.Tensor, class_idx: int = None) -> np.ndarray:
        """
        x: (1, 3, H, W) normalized input tensor
        Returns: (H, W) numpy array in [0, 1]
        """
        self.model.eval()
        self.model.zero_grad()

        logits = self.model(x, gate_alpha=1.0)          # forward pass
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        score = logits[0, class_idx]
        score.backward()                                  # backward pass

        # gradients: (1, N, C), activations: (1, N, C)
        grads = self.gradients[0]          # (N, C)
        acts  = self.activations[0]        # (N, C)

        # Global average pool over channels → weights (N,)
        weights = grads.mean(dim=-1)       # (N,)

        # Weighted combination of activations
        cam = (weights.unsqueeze(-1) * acts).sum(dim=-1)  # (N,)
        cam = F.relu(cam)                                   # ReLU

        # Reshape tokens → spatial map
        # For Swin-S stage 4: 7×7 = 49 tokens
        N = cam.shape[0]
        spatial = int(math.isqrt(N))
        if spatial * spatial != N:
            # Non-square token count: pad to nearest square
            spatial = int(math.ceil(N ** 0.5))

        cam_2d = cam.cpu().float().numpy()
        try:
            cam_2d = cam_2d[:spatial*spatial].reshape(spatial, spatial)
        except Exception:
            cam_2d = cam_2d.reshape(1, -1)

        # Resize to input resolution
        cam_resized = cv2.resize(cam_2d, (IMG_SIZE, IMG_SIZE),
                                  interpolation=cv2.INTER_LINEAR)

        # Normalize to [0, 1]
        cam_min = cam_resized.min()
        cam_max = cam_resized.max()
        if cam_max - cam_min > 1e-8:
            cam_resized = (cam_resized - cam_min) / (cam_max - cam_min)
        else:
            cam_resized = np.zeros_like(cam_resized)

        return cam_resized.astype(np.float32)


def get_target_layer(model: nn.Module) -> nn.Module:
    """
    Returns the final Swin stage norm1 layer.
    This is the same hook point used in your existing XAI pipeline.

    For DataParallel-wrapped models, access via model.module.
    For Swin-S: backbone.layers[3].blocks[-1].norm1
    """
    # Unwrap DataParallel if needed
    base = model.module if isinstance(model, nn.DataParallel) else model
    try:
        # Standard timm Swin attribute path
        layer = base.backbone.layers[3].blocks[-1].norm1
        print(f"✅ Target layer found: backbone.layers[3].blocks[-1].norm1")
        return layer
    except AttributeError:
        pass

    # Fallback: walk named modules and find the last LayerNorm
    last_ln = None
    for name, module in base.named_modules():
        if isinstance(module, nn.LayerNorm):
            last_ln = module
    if last_ln is not None:
        print(f"⚠️  Using fallback: last LayerNorm in model")
        return last_ln

    raise RuntimeError(
        "Could not find target layer. "
        "Check the Swin attribute path for your timm version."
    )


def load_model(ckpt_path: str, device: str) -> nn.Module:
    """
    Loads SwinRGBHSV (HSV-raw, scalar gate) from checkpoint.
    Handles both full checkpoints (with 'model' key) and
    weights-only files (ema_model_weights_only.pth).
    """
    print(f"\n📂 Loading checkpoint: {ckpt_path}")

    model = SwinRGBHSV(
        model_name     = "swin_small_patch4_window7_224",
        pretrained     = False,   # weights come from checkpoint
        drop_path_rate = 0.2,
        num_classes    = 7,
        use_hsv_branch = True,
        hsv_embed_dim  = 128,
        hsv_use_sincos = True, 
        hsv_dropout    = 0.1,
        fuse_dropout   = 0.2,
        gate_hidden    = 256,
        gate_vector    = True,   
    )

    ckpt = torch.load(ckpt_path, map_location="cpu")

    # Determine what kind of checkpoint this is
    if isinstance(ckpt, dict) and "model" in ckpt:
        # Full checkpoint: may contain EMA shadow
        if "ema_shadow" in ckpt:
            print("   Found EMA shadow — merging EMA weights into model state")
            state = dict(ckpt["model"])
            for k, v in ckpt["ema_shadow"].items():
                if k in state:
                    state[k] = v
        else:
            state = ckpt["model"]
    else:
        # Weights-only file (ema_model_weights_only.pth)
        state = ckpt

    # Strip DataParallel prefix if present
    state = {(k[7:] if k.startswith("module.") else k): v for k, v in state.items()}

    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f"   ⚠️  Missing keys  ({len(missing)}): {missing[:5]}...")
    if unexpected:
        print(f"   ⚠️  Unexpected keys ({len(unexpected)}): {unexpected[:5]}...")
    if not missing and not unexpected:
        print("   ✅ Weights loaded cleanly (strict=True equivalent)")

    model.eval()
    model.to(device)
    print(f"   Model on: {device}")
    return model


def generate_mask_for_image(
    gradcam: GradCAM,
    img_path: Path,
    preprocess: T.Compose,
    device: str,
    threshold_pct: int = 70,
    morph_kernel: int = 5,
) -> np.ndarray:
    """
    Generates a binary pseudo-mask for one image.

    Returns: (224, 224) uint8 array — 255 = lesion, 0 = background
    """
    img_pil    = Image.open(img_path).convert("RGB")
    img_tensor = preprocess(img_pil).unsqueeze(0).to(device)

    # GradCAM heatmap: (224, 224) float in [0, 1]
    heatmap = gradcam(img_tensor, class_idx=None)

    # Threshold at Nth percentile
    thresh_val   = np.percentile(heatmap, threshold_pct)
    binary_mask  = (heatmap >= thresh_val).astype(np.uint8) * 255

    # Optional morphological cleanup:
    # - OPEN removes small isolated noise blobs
    # - CLOSE fills small holes within lesion regions
    if morph_kernel > 0:
        kernel      = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (morph_kernel, morph_kernel)
        )
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN,  kernel)
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)

    return binary_mask


def main():
    print("=" * 70)
    print("PHASE 1: PSEUDO-MASK GENERATION")
    print("=" * 70)
    print(f"Checkpoint : {CHECKPOINT_PATH}")
    print(f"Data root  : {DATA_ROOT}/train")
    print(f"Output dir : {MASK_OUT_DIR}")
    print(f"Device     : {DEVICE}")
    print(f"Threshold  : {THRESHOLD_PCT}th percentile")
    print("=" * 70)

    # ── Verify checkpoint exists ───────────────────────────────────────────
    ckpt_path = Path(CHECKPOINT_PATH)
    if not ckpt_path.exists():
        # Try alternative path
        alt = Path("/kaggle/temp/swin_small_hsv_raw_seed2026/best_model.pth")
        if alt.exists():
            print(f"⚠️  Primary path not found. Using: {alt}")
            ckpt_path = alt
        else:
            raise FileNotFoundError(
                f"Checkpoint not found at:\n  {CHECKPOINT_PATH}\n  {alt}\n"
                "Check your experiment output directory."
            )

    # ── Load model ────────────────────────────────────────────────────────
    model = load_model(str(ckpt_path), DEVICE)

    # ── Get GradCAM target layer ──────────────────────────────────────────
    target_layer = get_target_layer(model)
    gradcam      = GradCAM(model, target_layer)

    # ── Preprocessing (eval pipeline — no augmentation) ──────────────────
    preprocess = T.Compose([
        T.Resize(int(IMG_SIZE * 1.14),
                 interpolation=InterpolationMode.BICUBIC),
        T.CenterCrop(IMG_SIZE),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # ── Walk train split ──────────────────────────────────────────────────
    train_root = Path(DATA_ROOT) / "train"
    if not train_root.exists():
        raise FileNotFoundError(f"Train split not found: {train_root}")

    class_dirs = sorted([d for d in train_root.iterdir() if d.is_dir()])
    print(f"\nFound {len(class_dirs)} class directories: "
          f"{[d.name for d in class_dirs]}")

    total_generated = 0
    total_skipped   = 0
    total_failed    = 0

    for cls_dir in class_dirs:
        out_cls_dir = Path(MASK_OUT_DIR) / cls_dir.name
        out_cls_dir.mkdir(parents=True, exist_ok=True)

        img_files = sorted(
            f for f in cls_dir.iterdir()
            if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        )

        print(f"\n  [{cls_dir.name}]  {len(img_files)} images")

        for img_path in tqdm(img_files, desc=f"  {cls_dir.name}", leave=False):
            mask_path = out_cls_dir / (img_path.stem + "_mask.png")

            # Skip if already exists (resume-friendly)
            if SKIP_EXISTING and mask_path.exists():
                total_skipped += 1
                continue

            try:
                mask = generate_mask_for_image(
                    gradcam      = gradcam,
                    img_path     = img_path,
                    preprocess   = preprocess,
                    device       = DEVICE,
                    threshold_pct= THRESHOLD_PCT,
                    morph_kernel = MORPH_KERNEL,
                )
                Image.fromarray(mask).save(str(mask_path))
                total_generated += 1

            except Exception as e:
                print(f"\n    ⚠️  Failed: {img_path.name} — {e}")
                total_failed += 1

    print("\n" + "=" * 70)
    print(f"✅ Done!")
    print(f"   Generated : {total_generated}")
    print(f"   Skipped   : {total_skipped}  (already existed)")
    print(f"   Failed    : {total_failed}")
    print(f"   Output    : {MASK_OUT_DIR}")
    print("=" * 70)

    # ── Quick sanity check: display 3 sample mask overlays ───────────────
    print("\n🔍 Sanity check: loading 3 random masks...")
    mask_files = list(Path(MASK_OUT_DIR).rglob("*_mask.png"))
    if len(mask_files) >= 3:
        for mf in random.sample(mask_files, 3):
            m = np.array(Image.open(mf))
            coverage = (m > 0).mean() * 100
            print(f"   {mf.name}: shape={m.shape}, "
                  f"coverage={coverage:.1f}%, "
                  f"unique_vals={np.unique(m).tolist()}")
    else:
        print(f"   Only {len(mask_files)} masks found — check for errors above.")


if __name__ == "__main__":
    main()

Overwriting generate_pseudo_masks.py


In [2]:
!pip install grad-cam opencv-python-headless -q
!python generate_pseudo_masks.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 63.7 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
PHASE 1: PSEUDO-MASK GENERATION
Checkpoint : /kaggle/working/experiments_swin/swin_small_hsv_vector_seed1337/model/ema_model_weights_only.pth
Data root  : /kaggle/input/tea-leaf701515/tea_leaf_processed_dataset/tea_leaf_processed_dataset/train
Output dir : /kaggle/working/pseudo_masks/train
Device     : cuda
Threshold  : 70th percentile

📂 Loading checkpoint: /kaggle/working/experiments_swin/swin_small_hsv_vector_seed1337/model/ema_model_weights_only.pth
   ✅ Weights loaded cleanly (strict=True equivalent)
   Model on: cuda
✅ Target layer found: backbone.layers[3].blocks[-1].norm1

Found 7 class directories: ['Brown Blight', 'Gray Blight', 'Green mirid bug', 'Healthy leaf', 'Helopeltis', 'Red spider', 'Tea algal leaf spot']

  [Brown Blight]  858 images
            

In [2]:
%%writefile train_tcca.py  
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import re
import csv
import json
import time
import math
import shutil
import random
import argparse
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List, ContextManager, Union
from collections import Counter
from contextlib import nullcontext

import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.datasets import ImageFolder
from PIL import Image

import timm
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)

# ── Import shared utilities from your existing Swin script ────────────────────
# This avoids code duplication and ensures HSV conversion is identical.
try:
    from train_swin_three_models import (
        set_seed, worker_init_fn, get_autocast_ctx, make_grad_scaler,
        rgb_to_hsv_torch, hsv_to_sincos_sv, normalize_hsv_rep,
        count_params_m, try_get_gflops, _json_safe_scalar,
        strip_module_prefix, add_module_prefix, load_state_dict_robust,
        EMA, CheckpointManager,
    )
    print("✅ Imported utilities from train_swin_three_models.py")
except ImportError:
    raise ImportError(
        "train_swin_three_models.py must be in the same directory.\n"
        "Copy it alongside this script before running."
    )


# =============================================================================
# TCCA MODULE
# =============================================================================

class ChromaticCrossAttention(nn.Module):
    """
    Token-Level Chromatic Cross-Attention.

    RGB token sequence is the Query source.
    Color feature map tokens are the Key/Value source.

    Shape contract:
      rgb_tokens  : (B, N, D)   — flattened Swin stage tokens
      color_tokens: (B, N, D)   — projected color feature map tokens
    Returns:
      attended    : (B, N, D)   — LayerNorm applied after attention

    Uses PyTorch's built-in MultiheadAttention for correctness and
    efficiency (fused attention kernel on CUDA).
    """

    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f"embed_dim={embed_dim} must be divisible by num_heads={num_heads}"

        self.attn = nn.MultiheadAttention(
            embed_dim   = embed_dim,
            num_heads   = num_heads,
            dropout     = dropout,
            batch_first = True,   # (B, N, D) convention throughout
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, rgb_tokens: torch.Tensor,
                color_tokens: torch.Tensor) -> torch.Tensor:
        # rgb_tokens  : (B, N, D)  → Query
        # color_tokens: (B, N, D)  → Key, Value
        attended, _ = self.attn(
            query = rgb_tokens,
            key   = color_tokens,
            value = color_tokens,
        )
        # Residual + LayerNorm (pre-norm style for stability)
        return self.norm(rgb_tokens + attended)


class ColorEncoder(nn.Module):
    """
    Lightweight CNN encoder producing spatial feature maps at two resolutions
    matching Swin-S stage 3 (14×14) and stage 4 (7×7).

    Input  : (B, in_channels, 224, 224)  HSV or LAB color image
    Outputs: feat3 (B, base_dim*2, 14, 14)
             feat4 (B, base_dim*4,  7,  7)

    base_dim=64 gives 128 and 256 channels respectively,
    matching the projection dimensions below.
    """

    def __init__(self, in_channels: int = 3, base_dim: int = 64):
        super().__init__()
        # Stride-2 convolutions to reach 56×56
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_dim, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim), nn.GELU(),
        )
        # 56×56 → 28×28
        self.layer1 = nn.Sequential(
            nn.Conv2d(base_dim, base_dim, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim), nn.GELU(),
        )
        # 28×28 → 14×14  (matches Swin-S stage 3 spatial resolution)
        self.layer2 = nn.Sequential(
            nn.Conv2d(base_dim, base_dim * 2, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim * 2), nn.GELU(),
        )
        # 14×14 → 7×7  (matches Swin-S stage 4 spatial resolution)
        self.layer3 = nn.Sequential(
            nn.Conv2d(base_dim * 2, base_dim * 4, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(base_dim * 4), nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.stem(x)    # (B,  64, 112, 112)
        x = self.layer1(x)  # (B,  64,  56,  56)
        f3 = self.layer2(x) # (B, 128,  14,  14)
        f4 = self.layer3(f3)# (B, 256,   7,   7)
        return f3, f4


class TCCAFusion(nn.Module):
    """
    Token-Level Chromatic Cross-Attention Fusion.

    Projects ColorEncoder outputs into Swin token space,
    then applies gated ChromaticCrossAttention at:
      - Stage 3 output: 14×14 tokens, D=384
      - Stage 4 output:  7× 7 tokens, D=768

    Gate scalar per stage, initialized conservatively.
    Warmup ramp applied externally (same gate_alpha as HSV branch).
    """

    def __init__(
        self,
        swin_dim_stage3: int  = 384,   # Swin-S stage 3 output dim
        swin_dim_stage4: int  = 768,   # Swin-S stage 4 output dim
        color_dim_stage3: int = 128,   # ColorEncoder feat3 channels
        color_dim_stage4: int = 256,   # ColorEncoder feat4 channels
        num_heads: int        = 8,
        attn_dropout: float   = 0.0,
    ):
        super().__init__()

        # Linear projections: color channels → Swin token dim
        self.proj3 = nn.Linear(color_dim_stage3, swin_dim_stage3)
        self.proj4 = nn.Linear(color_dim_stage4, swin_dim_stage4)

        self.tcca3 = ChromaticCrossAttention(swin_dim_stage3, num_heads, attn_dropout)
        self.tcca4 = ChromaticCrossAttention(swin_dim_stage4, num_heads, attn_dropout)

        # Per-stage scalar gates, initialized to -2.0 → sigmoid(-2) ≈ 0.12
        # This mirrors the conservative initialization in your existing code.
        self.gate3 = nn.Parameter(torch.full((1,), -2.0))
        self.gate4 = nn.Parameter(torch.full((1,), -2.0))

    def forward(
        self,
        rgb_tokens3:  torch.Tensor,   # (B, 196, 384)  — 14×14 = 196 tokens
        rgb_tokens4:  torch.Tensor,   # (B,  49, 768)  —  7× 7 =  49 tokens
        color_feat3:  torch.Tensor,   # (B, 128,  14, 14)
        color_feat4:  torch.Tensor,   # (B, 256,   7,  7)
        gate_alpha:   float = 1.0,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B = rgb_tokens3.shape[0]

        # Flatten spatial dims: (B, C, H, W) → (B, H*W, C)
        c3 = color_feat3.flatten(2).transpose(1, 2)   # (B, 196, 128)
        c4 = color_feat4.flatten(2).transpose(1, 2)   # (B,  49, 256)

        # Project to Swin dims
        c3 = self.proj3(c3)    # (B, 196, 384)
        c4 = self.proj4(c4)    # (B,  49, 768)

        # Cast color tokens to same dtype as RGB tokens (handles AMP fp16)
        c3 = c3.to(dtype=rgb_tokens3.dtype)
        c4 = c4.to(dtype=rgb_tokens4.dtype)

        # Gated residual: f_out = f_rgb + alpha * sigmoid(gate) * TCCA(f_rgb, c)
        g3 = gate_alpha * torch.sigmoid(self.gate3)
        g4 = gate_alpha * torch.sigmoid(self.gate4)

        out3 = rgb_tokens3 + g3 * self.tcca3(rgb_tokens3, c3)
        out4 = rgb_tokens4 + g4 * self.tcca4(rgb_tokens4, c4)

        return out3, out4


# =============================================================================
# FPN SEGMENTATION DECODER
# =============================================================================

class FPNSegDecoder(nn.Module):
    """
    Lightweight FPN-style segmentation decoder.

    Consumes Swin stage 1, 2, 3 feature maps (spatial conv features
    reconstructed from token sequences) and produces a binary lesion
    mask at 224×224 resolution.

    Only active during training when pseudo-masks are provided.
    Returns logits (before sigmoid); sigmoid applied in loss function.
    """

    def __init__(
        self,
        stage_dims: List[int] = [96, 192, 384],  # Swin-S stages 1–3
        decoder_dim: int      = 128,
        num_classes: int      = 1,                # binary lesion mask
    ):
        super().__init__()

        # Lateral projections: reduce each stage to decoder_dim
        self.lat1 = nn.Conv2d(stage_dims[0], decoder_dim, 1)
        self.lat2 = nn.Conv2d(stage_dims[1], decoder_dim, 1)
        self.lat3 = nn.Conv2d(stage_dims[2], decoder_dim, 1)

        # Top-down upsampling path
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(decoder_dim, decoder_dim, 2, stride=2),
            nn.BatchNorm2d(decoder_dim), nn.GELU(),
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(decoder_dim, decoder_dim, 2, stride=2),
            nn.BatchNorm2d(decoder_dim), nn.GELU(),
        )
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(decoder_dim, decoder_dim // 2, 2, stride=2),
            nn.BatchNorm2d(decoder_dim // 2), nn.GELU(),
        )

        # Final upsample from 56×56 to 224×224 (4× bilinear)
        self.final_up = nn.Upsample(
            scale_factor=2, mode="bilinear", align_corners=False
        )
        self.head = nn.Conv2d(decoder_dim // 2, num_classes, 1)

    def forward(
        self,
        f1: torch.Tensor,   # (B,  96, 56, 56)  Stage 1
        f2: torch.Tensor,   # (B, 192, 28, 28)  Stage 2
        f3: torch.Tensor,   # (B, 384, 14, 14)  Stage 3
    ) -> torch.Tensor:      # (B,   1, 224, 224) logits

        p3 = self.lat3(f3)                   # (B, 128, 14, 14)
        p2 = self.lat2(f2) + self.up3(p3)    # (B, 128, 28, 28)
        p1 = self.lat1(f1) + self.up2(p2)    # (B, 128, 56, 56)
        out = self.up1(p1)                   # (B,  64, 112, 112)
        out = self.final_up(out)             # (B,  64, 224, 224)
        return self.head(out)                # (B,   1, 224, 224)


# =============================================================================
# SWIN FEATURE EXTRACTOR WITH HOOKS
# =============================================================================

class SwinWithHooks(nn.Module):
    """
    Wraps a timm Swin backbone and extracts intermediate stage outputs
    via forward hooks — NO modification to timm internals.

    Captured outputs (after each stage's patch_merging / norm):
      stage1_out: (B,  96, 56, 56)   — reshaped from (B, 3136, 96)
      stage2_out: (B, 192, 28, 28)   — reshaped from (B,  784, 192)
      stage3_out: (B, 384, 14, 14)   — reshaped from (B,  196, 384)
      stage4_out: (B, 768,  7,  7)   — reshaped from (B,   49, 768)

    For TCCA we need stage3 and stage4 as TOKEN sequences (B, N, D).
    For the segmentation FPN we need stages 1–3 as SPATIAL maps (B, C, H, W).
    Both are derived from the same hooks.
    """

    # Swin-S spatial resolutions after each stage
    STAGE_SPATIAL = {0: (56, 56), 1: (28, 28), 2: (14, 14), 3: (7, 7)}
    STAGE_DIMS    = {0: 96, 1: 192, 2: 384, 3: 768}   # Swin-S/T channel dims

    def __init__(self, timm_id: str, pretrained: bool, drop_path_rate: float):
        super().__init__()
        self.swin = timm.create_model(
            timm_id,
            pretrained     = pretrained,
            num_classes    = 0,
            drop_path_rate = drop_path_rate,
            global_pool    = "",   # CRITICAL: disable GAP so we get token output
        )
        self._stage_outputs: Dict[int, torch.Tensor] = {}
        self._register_stage_hooks()

    def _register_stage_hooks(self):
        """
        Register forward hooks on each of the 4 Swin stages.
        timm Swin stores stages in self.swin.layers (a ModuleList of length 4).
        Each layer outputs (B, N, C) tokens.
        """
        for stage_idx in range(4):
            stage = self.swin.layers[stage_idx]

            def make_hook(idx):
                def hook(module, input, output):
                    # output: (B, N, C)  — token sequence
                    # Store as-is; reshape to spatial in forward() as needed
                    self._stage_outputs[idx] = output
                return hook

            stage.register_forward_hook(make_hook(stage_idx))

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Args:
            x: (B, 3, 224, 224) ImageNet-normalized RGB

        Returns dict with:
            tokens3   : (B, 196, 384)    Stage 3 tokens    — for TCCA
            tokens4   : (B,  49, 768)    Stage 4 tokens    — for TCCA
            feat1     : (B,  96, 56, 56) Stage 1 spatial   — for FPN
            feat2     : (B, 192, 28, 28) Stage 2 spatial   — for FPN
            feat3     : (B, 384, 14, 14) Stage 3 spatial   — for FPN
            pooled    : (B, 768)          Global avg pool   — for classifier

        Handles all timm Swin output formats:
            (B, N, C)    — older timm  (already tokens)
            (B, H, W, C) — newer timm  (channels-last spatial, most common on Kaggle)
            (B, C, H, W) — channels-first spatial (rare)
        """
        self._stage_outputs.clear()

        # Run full Swin forward (hooks populate _stage_outputs automatically)
        _ = self.swin(x)

        B = x.shape[0]

        def normalize_to_tokens(t: torch.Tensor) -> torch.Tensor:
            if t.dim() == 3:
                return t                          # (B, N, C) already
            elif t.dim() == 4:
                if t.shape[-1] < t.shape[1]:
                    B_, C, H, W = t.shape         # (B, C, H, W)
                    return t.permute(0, 2, 3, 1).reshape(B_, H * W, C)
                else:
                    B_, H, W, C = t.shape         # (B, H, W, C)  ← your timm version
                    return t.reshape(B_, H * W, C)
            else:
                raise ValueError(f"Unexpected shape: {t.shape}")

        def tokens_to_spatial(tokens: torch.Tensor) -> torch.Tensor:
            B_, N, C = tokens.shape
            H = W = int(N ** 0.5)
            assert H * W == N, f"Non-square token count N={N}"
            return tokens.transpose(1, 2).reshape(B_, C, H, W)

        s0 = normalize_to_tokens(self._stage_outputs[0])
        s1 = normalize_to_tokens(self._stage_outputs[1])
        s2 = normalize_to_tokens(self._stage_outputs[2])
        s3 = normalize_to_tokens(self._stage_outputs[3])

        feat1 = tokens_to_spatial(s0)
        feat2 = tokens_to_spatial(s1)
        feat3 = tokens_to_spatial(s2)

        tokens3 = s2
        tokens4 = s3
        pooled  = s3.mean(dim=1)

        return {
            "tokens3": tokens3,
            "tokens4": tokens4,
            "feat1":   feat1,
            "feat2":   feat2,
            "feat3":   feat3,
            "pooled":  pooled,
        }


# =============================================================================
# FULL MODEL: Swin + TCCA + optional HSV + optional SegHead
# =============================================================================

class SwinTCCA(nn.Module):
    """
    Swin-S with Token-Level Chromatic Cross-Attention fusion.

    Modes (controlled by constructor flags):
      use_hsv=False, use_seg=False  →  RGB + TCCA  (Phase 3a)
      use_hsv=True,  use_seg=False  →  RGB + HSV + TCCA  (Phase 3b)
      use_hsv=True,  use_seg=True   →  RGB + HSV + TCCA + Seg  (Phase 3c)

    The TCCA module is always present when this model is instantiated —
    it is the defining architectural contribution of this training phase.
    """

    SWIN_ID = "swin_small_patch4_window7_224.ms_in1k"

    def __init__(
        self,
        pretrained:      bool  = True,
        drop_path_rate:  float = 0.2,
        num_classes:     int   = 7,
        use_hsv:         bool  = False,
        hsv_use_sincos:  bool  = False,   # False = raw HSV (your best config)
        use_seg:         bool  = False,
        # TCCA hyperparams
        tcca_num_heads:  int   = 8,
        tcca_dropout:    float = 0.0,
        # HSV branch hyperparams (only used when use_hsv=True)
        color_base_dim:  int   = 64,      # ColorEncoder base_dim
        # Classifier head
        fuse_dropout:    float = 0.2,
        # ImageNet stats (for HSV de-normalization)
        img_mean: Tuple[float,...] = (0.485, 0.456, 0.406),
        img_std:  Tuple[float,...] = (0.229, 0.224, 0.225),
    ):
        super().__init__()
        self.use_hsv        = use_hsv
        self.hsv_use_sincos = hsv_use_sincos
        self.use_seg        = use_seg
        self.num_classes    = num_classes

        self.register_buffer("img_mean", torch.tensor(img_mean).view(1, 3, 1, 1))
        self.register_buffer("img_std",  torch.tensor(img_std).view(1, 3, 1, 1))

        # ── Swin backbone with stage hooks ────────────────────────────────
        self.swin = SwinWithHooks(
            timm_id        = self.SWIN_ID,
            pretrained     = pretrained,
            drop_path_rate = drop_path_rate,
        )

        # ── Color encoder and TCCA ─────────────────────────────────────────
        # When use_hsv=False, we still run TCCA but feed it zeros —
        # effectively making it learn to ignore the color branch.
        # A cleaner approach: still instantiate ColorEncoder but feed
        # a zero tensor, so gradients flow normally and gate learns ~0.
        # Actually: when use_hsv=False, we skip the color encoder entirely
        # and run TCCA in "RGB self-attention" mode (queries RGB tokens,
        # keys/values are also from RGB). This tests whether cross-attention
        # itself is beneficial independent of color.
        in_ch = 4 if (use_hsv and hsv_use_sincos) else 3

        if use_hsv:
            self.color_encoder = ColorEncoder(
                in_channels = in_ch,
                base_dim    = color_base_dim,
            )
            color_dim3 = color_base_dim * 2   # 128
            color_dim4 = color_base_dim * 4   # 256
        else:
            # RGB self-cross-attention: project stage outputs to serve as
            # "color" keys/values. Uses same spatial resolution.
            self.color_proj3 = nn.Linear(384, 384)   # Stage 3 self
            self.color_proj4 = nn.Linear(768, 768)   # Stage 4 self
            color_dim3 = 384
            color_dim4 = 768

        self.tcca = TCCAFusion(
            swin_dim_stage3  = 384,
            swin_dim_stage4  = 768,
            color_dim_stage3 = color_dim3,
            color_dim_stage4 = color_dim4,
            num_heads        = tcca_num_heads,
            attn_dropout     = tcca_dropout,
        )

        # ── Segmentation head (Phase 3c only) ─────────────────────────────
        if use_seg:
            self.seg_decoder = FPNSegDecoder(
                stage_dims  = [96, 192, 384],
                decoder_dim = 128,
                num_classes = 1,
            )

        # ── Classification head ───────────────────────────────────────────
        # Two-layer MLP on the TCCA-fused Stage 4 pooled feature
        self.classifier = nn.Sequential(
            nn.Dropout(fuse_dropout),
            nn.Linear(768, 384),
            nn.GELU(),
            nn.Dropout(fuse_dropout),
            nn.Linear(384, num_classes),
        )

    def _get_color_tokens(
        self,
        x_norm:   torch.Tensor,
        swin_out: Dict[str, torch.Tensor],
        gate_alpha: float,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns (color_tokens3, color_tokens4) in SPATIAL map form
        (B, C, H, W) for TCCAFusion to project and flatten.
        """
        if self.use_hsv:
            # De-normalize to [0, 1] in FP32 with AMP disabled
            autocast_ctx = (
                torch.cuda.amp.autocast(enabled=False)
                if torch.cuda.is_available() else nullcontext()
            )
            with autocast_ctx:
                x_raw = (
                    x_norm.float() * self.img_std.float()
                    + self.img_mean.float()
                ).clamp(0.0, 1.0)
                x_hsv = rgb_to_hsv_torch(x_raw)
                if self.hsv_use_sincos:
                    x_color = normalize_hsv_rep(hsv_to_sincos_sv(x_hsv))
                else:
                    x_color = x_hsv   # raw HSV [0, 1]

            x_color = x_color.to(dtype=swin_out["tokens3"].dtype)
            feat3, feat4 = self.color_encoder(x_color)  # spatial maps
            return feat3, feat4
        else:
            B = x_norm.shape[0]
            proj3_device = self.color_proj3.weight.device
            proj4_device = self.color_proj4.weight.device
            c3 = self.color_proj3(swin_out["tokens3"].to(proj3_device))
            c4 = self.color_proj4(swin_out["tokens4"].to(proj4_device))
            feat3 = c3.transpose(1, 2).reshape(B, 384, 14, 14)
            feat4 = c4.transpose(1, 2).reshape(B, 768,  7,  7)
            return feat3, feat4

    def forward(
        self,
        x_norm:     torch.Tensor,
        gate_alpha: float = 1.0,
        return_seg: bool  = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """
        Args:
            x_norm    : (B, 3, 224, 224) ImageNet-normalized RGB
            gate_alpha: warmup scalar in [0, 1]
            return_seg: if True AND use_seg=True, returns (cls_logits, seg_logits)

        Returns:
            cls_logits: (B, num_classes)
            seg_logits: (B, 1, 224, 224)  [only when return_seg=True]
        """
        # ── Step 1: Swin forward pass (hooks capture all stage outputs) ────
        swin_out = self.swin(x_norm)
        # swin_out keys: tokens3, tokens4, feat1, feat2, feat3, pooled

        # ── Step 2: Get color tokens ───────────────────────────────────────
        color_feat3, color_feat4 = self._get_color_tokens(
            x_norm, swin_out, gate_alpha
        )

        # ── Step 3: TCCA fusion at Stage 3 and Stage 4 ────────────────────
        fused_tokens3, fused_tokens4 = self.tcca(
            rgb_tokens3 = swin_out["tokens3"],
            rgb_tokens4 = swin_out["tokens4"],
            color_feat3 = color_feat3,
            color_feat4 = color_feat4,
            gate_alpha  = gate_alpha,
        )

        # ── Step 4: Classification — GAP on fused Stage 4 tokens ──────────
        cls_feat   = fused_tokens4.mean(dim=1)   # (B, 768) — global avg pool
        cls_logits = self.classifier(cls_feat)    # (B, num_classes)

        # ── Step 5: Segmentation (Phase 3c only) ──────────────────────────
        if return_seg and self.use_seg:
            # Use fused Stage 3 tokens reshaped back to spatial
            B = x_norm.shape[0]
            fused_feat3 = fused_tokens3.transpose(1, 2).reshape(B, 384, 14, 14)
            seg_logits  = self.seg_decoder(
                f1 = swin_out["feat1"],    # (B, 96,  56, 56)
                f2 = swin_out["feat2"],    # (B, 192, 28, 28)
                f3 = fused_feat3,          # (B, 384, 14, 14) — TCCA fused
            )
            return cls_logits, seg_logits

        return cls_logits


# =============================================================================
# MULTI-TASK DATASET  (handles missing masks gracefully)
# =============================================================================

class TeaLeafDataset(Dataset):
    """
    Drop-in replacement for ImageFolder that additionally loads pseudo-masks.

    For val/test splits or images without masks, mask=None is returned.
    The training loop handles None masks by skipping the seg loss contribution
    for those samples.
    """

    CLASS_NAMES = [
        "Brown Blight", "Gray Blight", "Green mirid bug",
        "Healthy leaf",  "Helopeltis",  "Red spider",
        "Tea algal leaf spot",
    ]

    def __init__(
        self,
        root_dir:    str,
        mask_dir:    Optional[str] = None,
        split:       str           = "train",
        img_size:    int           = 224,
        augment:     bool          = True,
        disable_hue_jitter: bool   = True,   # always True when HSV branch active
    ):
        self.root_dir  = Path(root_dir)
        self.mask_dir  = Path(mask_dir) if mask_dir else None
        self.split     = split
        self.augment   = augment and (split == "train")

        # Build class → index map from sorted directory listing
        # (matches ImageFolder ordering)
        self.class_to_idx = {c: i for i, c in enumerate(self.CLASS_NAMES)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}

        # Discover all images
        self.samples: List[Tuple[Path, int]] = []
        for cls_name in self.CLASS_NAMES:
            cls_path = self.root_dir / cls_name
            if not cls_path.exists():
                continue
            for img_path in sorted(cls_path.glob("*")):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                    self.samples.append((img_path, self.class_to_idx[cls_name]))

        # Also accept ImageFolder-style numeric subdirs (fallback)
        if len(self.samples) == 0:
            # Try using ImageFolder to discover samples
            tmp = ImageFolder(str(self.root_dir))
            for path, lbl in tmp.samples:
                self.samples.append((Path(path), lbl))

        if len(self.samples) == 0:
            raise RuntimeError(f"No images found in {self.root_dir}")

        # Expose .targets for DataLoader sampler compatibility
        self.targets = [lbl for _, lbl in self.samples]
        self.classes = self.CLASS_NAMES

        # Transforms
        hue = 0.0 if disable_hue_jitter else 0.1
        if self.augment:
            self.img_transform = T.Compose([
                T.RandomResizedCrop(img_size, scale=(0.85, 1.0),
                                    interpolation=InterpolationMode.BICUBIC),
                T.RandomHorizontalFlip(p=0.5),
                T.ColorJitter(brightness=0.2, contrast=0.2,
                              saturation=0.2, hue=hue),
                T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])
        else:
            self.img_transform = T.Compose([
                T.Resize(int(img_size * 1.14),
                         interpolation=InterpolationMode.BICUBIC),
                T.CenterCrop(img_size),
                T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])

        self.mask_resize = T.Compose([
            T.Resize(img_size, interpolation=InterpolationMode.NEAREST),
            T.CenterCrop(img_size),
        ])

        self.img_size = img_size

    def _load_mask(self, img_path: Path) -> Optional[torch.Tensor]:
        """Returns (1, H, W) float32 tensor in [0,1], or None."""
        if self.mask_dir is None:
            return None
        try:
            rel      = img_path.relative_to(self.root_dir)
            mask_p   = self.mask_dir / rel.parent / (rel.stem + "_mask.png")
            if not mask_p.exists():
                return None
            mask_pil = Image.open(mask_p).convert("L")
            mask_pil = self.mask_resize(mask_pil)
            mask_np  = np.array(mask_pil, dtype=np.float32) / 255.0
            return torch.from_numpy(mask_np).unsqueeze(0)   # (1, H, W)
        except Exception:
            return None

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict:
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.img_transform(img)
        mask_tensor = self._load_mask(img_path)
        return {
            "image": img_tensor,
            "label": torch.tensor(label, dtype=torch.long),
            "mask":  mask_tensor,   # (1,H,W) float32 or None
        }


def collate_with_masks(batch: List[Dict]) -> Dict:
    """
    Custom collate that handles variable mask presence.
    Stacks images and labels normally; keeps masks as a list (not a tensor)
    because some entries may be None.
    """
    images = torch.stack([b["image"] for b in batch])
    labels = torch.stack([b["label"] for b in batch])
    masks  = [b["mask"] for b in batch]   # list of (1,H,W) tensors or Nones
    return {"image": images, "label": labels, "mask": masks}


# =============================================================================
# MULTI-TASK LOSS
# =============================================================================

def dice_loss(pred: torch.Tensor, target: torch.Tensor,
              eps: float = 1e-6) -> torch.Tensor:
    """Soft Dice loss for binary segmentation. pred is raw logits."""
    p = torch.sigmoid(pred)
    inter = (p * target).sum(dim=(2, 3))
    union = p.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return (1.0 - (2.0 * inter + eps) / (union + eps)).mean()


def multitask_loss(
    cls_logits:      torch.Tensor,          # (B, K)
    seg_logits:      Optional[torch.Tensor],# (B, 1, H, W) or None
    labels:          torch.Tensor,          # (B,)
    masks:           List,                  # list length B: (1,H,W) or None
    lambda_cls:      float = 1.0,
    lambda_seg:      float = 0.5,
    label_smoothing: float = 0.1,
    device:          str   = "cuda",
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Returns (total_loss, log_dict).
    Segmentation loss is computed only for samples with valid pseudo-masks.
    Samples with mask=None contribute 0 to seg loss (no gradient there).
    """
    loss_cls = F.cross_entropy(cls_logits, labels,
                                label_smoothing=label_smoothing)
    log = {"loss_cls": loss_cls.item(), "loss_seg": 0.0, "n_seg": 0}

    loss_seg = torch.tensor(0.0, device=device)
    if seg_logits is not None:
        valid_idx = [i for i, m in enumerate(masks) if m is not None]
        if valid_idx:
            idx_t   = torch.tensor(valid_idx, device=device)
            mask_t  = torch.stack(
                [masks[i].to(device) for i in valid_idx]
            )                                          # (n, 1, H, W)
            seg_sub = seg_logits[idx_t]                # (n, 1, H, W)

            bce      = F.binary_cross_entropy_with_logits(
                seg_sub, mask_t, reduction="mean"
            )
            dice     = dice_loss(seg_sub, mask_t)
            loss_seg = bce + dice
            log["loss_seg"] = loss_seg.item()
            log["n_seg"]    = len(valid_idx)

    total = lambda_cls * loss_cls + lambda_seg * loss_seg
    log["loss_total"] = total.item()
    return total, log


# =============================================================================
# CONFIG
# =============================================================================

@dataclass
class TCCAConfig:
    # Model
    num_classes:     int   = 7
    pretrained:      bool  = True
    drop_path_rate:  float = 0.2
    use_hsv:         bool  = False
    hsv_use_sincos:  bool  = False   # False = raw HSV
    use_seg:         bool  = False
    tcca_num_heads:  int   = 8
    gate_warmup_epochs: int = 5

    # Multi-task loss
    lambda_cls:  float = 1.0
    lambda_seg:  float = 0.5

    # Data
    data_root:   str  = "/kaggle/input/tea-leaf701515/tea_leaf_processed_dataset/tea_leaf_processed_dataset"
    mask_dir:    str  = ""    # empty = no segmentation
    input_size:  int  = 224
    batch_size:  int  = 48    # slightly smaller than 64 due to TCCA memory overhead
    num_workers: int  = 2

    # Training
    epochs:                    int   = 80
    warmup_epochs:             int   = 5
    lr:                        float = 5e-4
    warmup_lr_init:            float = 1e-6
    weight_decay:              float = 0.05
    min_lr:                    float = 1e-6
    label_smoothing:           float = 0.1
    grad_clip_norm:            float = 1.0
    gradient_accumulation_steps: int = 1
    use_amp:                   bool  = True
    use_ema:                   bool  = False
    ema_decay:                 float = 0.9998
    early_stopping_patience:   int   = 25
    compute_val_auc:           bool  = False

    # Logging
    run_dir:           str  = "/kaggle/working/experiments_tcca"
    experiment_name:   str  = "tcca_run"
    ckpt_temp_dir:     str  = "/kaggle/temp"
    log_interval:      int  = 50
    save_cm_png:       bool = True
    save_epoch_checkpoints: bool = False
    save_epoch_every:  int  = 5
    keep_last_n_checkpoints: int = 3

    # System
    device:           str  = "cuda" if torch.cuda.is_available() else "cpu"
    seed:             int  = 42
    deterministic:    bool = True
    use_data_parallel: bool = True

    def validate(self):
        if not Path(self.data_root).exists():
            raise FileNotFoundError(f"data_root not found: {self.data_root}")
        if self.use_seg and not self.mask_dir:
            raise ValueError(
                "--use_seg requires --mask_dir pointing to pseudo-mask directory."
            )
        if self.use_seg and self.mask_dir and not Path(self.mask_dir).exists():
            raise FileNotFoundError(f"mask_dir not found: {self.mask_dir}")
        if self.batch_size < 1:
            raise ValueError("batch_size must be >= 1")


# =============================================================================
# TRAINER
# =============================================================================

class TCCATrainer:

    def __init__(self, cfg: TCCAConfig):
        cfg.validate()
        self.cfg = cfg
        set_seed(cfg.seed, deterministic=cfg.deterministic)

        self.run_root   = Path(cfg.run_dir)
        self.run_root.mkdir(parents=True, exist_ok=True)
        self.exp_dir    = self.run_root / cfg.experiment_name
        self.exp_dir.mkdir(parents=True, exist_ok=True)
        self.logs_dir   = self.exp_dir / "logs"; self.logs_dir.mkdir(exist_ok=True)
        self.config_dir = self.exp_dir / "config"; self.config_dir.mkdir(exist_ok=True)

        self.ckpt_manager = CheckpointManager(
            temp_root       = Path(cfg.ckpt_temp_dir),
            final_root      = self.run_root,
            experiment_name = cfg.experiment_name,
        )

        # ── Build model ───────────────────────────────────────────────────
        print(f"\n🏗️  Building SwinTCCA")
        print(f"   use_hsv={cfg.use_hsv}  use_seg={cfg.use_seg}  "
              f"use_ema={cfg.use_ema}")
        self.model = SwinTCCA(
            pretrained     = cfg.pretrained,
            drop_path_rate = cfg.drop_path_rate,
            num_classes    = cfg.num_classes,
            use_hsv        = cfg.use_hsv,
            hsv_use_sincos = cfg.hsv_use_sincos,
            use_seg        = cfg.use_seg,
            tcca_num_heads = cfg.tcca_num_heads,
        ).to(cfg.device)

        self.params_m   = float(count_params_m(self.model))
        print(f"   Params: {self.params_m:.2f}M")

        # ── DataParallel ──────────────────────────────────────────────────
        self.n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
        if self.n_gpus > 1 and cfg.use_data_parallel:
            print(f"🚀 DataParallel: {self.n_gpus} GPUs")
            self.model = nn.DataParallel(self.model)

        base_model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        self.ema   = EMA(base_model, cfg.ema_decay) if cfg.use_ema else None

        # ── Data loaders ──────────────────────────────────────────────────
        self.train_loader, self.val_loader, self.test_loader = self._build_loaders()

        # ── Optimizer and scheduler ───────────────────────────────────────
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.999),
        )

        bpe             = len(self.train_loader)
        accum           = cfg.gradient_accumulation_steps
        upd_per_epoch   = int(math.ceil(bpe / accum))
        total_upd       = cfg.epochs * upd_per_epoch
        warmup_upd      = min(cfg.warmup_epochs * upd_per_epoch,
                               max(0, total_upd - 1))

        if warmup_upd > 0:
            self.scheduler = SequentialLR(
                self.optimizer,
                schedulers=[
                    LinearLR(self.optimizer,
                              start_factor=cfg.warmup_lr_init / cfg.lr,
                              end_factor=1.0, total_iters=warmup_upd),
                    CosineAnnealingLR(self.optimizer,
                                      T_max=max(1, total_upd - warmup_upd),
                                      eta_min=cfg.min_lr),
                ],
                milestones=[warmup_upd],
            )
        else:
            self.scheduler = CosineAnnealingLR(
                self.optimizer, T_max=max(1, total_upd), eta_min=cfg.min_lr
            )

        self.scaler      = make_grad_scaler(cfg.use_amp)
        self.best_val_f1 = -1.0
        self.bad_epochs  = 0
        self.start_epoch = 1
        self.train_log: List[Dict] = []

        self._save_config()
        self._print_data_stats()

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _save_config(self):
        with open(self.config_dir / "config.json", "w") as f:
            json.dump(self.cfg.__dict__, f, indent=2)

    def _print_data_stats(self):
        n_train = len(self.train_loader.dataset)
        n_val   = len(self.val_loader.dataset)
        n_test  = len(self.test_loader.dataset)
        print(f"\n📦 Data: Train={n_train} | Val={n_val} | Test={n_test}")
        if self.cfg.use_seg and self.cfg.mask_dir:
            mask_count = len(list(Path(self.cfg.mask_dir).rglob("*_mask.png")))
            print(f"   Pseudo-masks: {mask_count}")

    def _build_loaders(self):
        root     = Path(self.cfg.data_root)
        mask_dir = self.cfg.mask_dir if self.cfg.use_seg else None
        use_p    = (self.cfg.num_workers > 0) and (self.cfg.epochs >= 2)
        g        = torch.Generator().manual_seed(self.cfg.seed)

        # disable_hue_jitter=True when HSV branch is active
        dh = self.cfg.use_hsv

        train_ds = TeaLeafDataset(root / "train", mask_dir,
                                   split="train", img_size=self.cfg.input_size,
                                   augment=True, disable_hue_jitter=dh)
        val_ds   = TeaLeafDataset(root / "val",   None,
                                   split="val",   img_size=self.cfg.input_size,
                                   augment=False, disable_hue_jitter=dh)
        test_ds  = TeaLeafDataset(root / "test",  None,
                                   split="test",  img_size=self.cfg.input_size,
                                   augment=False, disable_hue_jitter=dh)

        train_loader = DataLoader(
            train_ds, batch_size=self.cfg.batch_size, shuffle=True,
            num_workers=self.cfg.num_workers, pin_memory=True,
            collate_fn=collate_with_masks,
            worker_init_fn=worker_init_fn, generator=g,
            persistent_workers=use_p,
        )
        val_loader = DataLoader(
            val_ds, batch_size=self.cfg.batch_size * 2, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=True,
            collate_fn=collate_with_masks, persistent_workers=use_p,
        )
        test_loader = DataLoader(
            test_ds, batch_size=self.cfg.batch_size * 2, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=True,
            collate_fn=collate_with_masks, persistent_workers=use_p,
        )
        return train_loader, val_loader, test_loader

    def _gate_alpha(self, epoch: int) -> float:
        w = max(1, self.cfg.gate_warmup_epochs)
        return float(min(1.0, epoch / w))

    def _base_model(self) -> SwinTCCA:
        return self.model.module if isinstance(self.model, nn.DataParallel) else self.model

    # ── Training ──────────────────────────────────────────────────────────────

    def train_one_epoch(self, epoch: int) -> Dict:
        self.model.train()
        total_loss, correct, n = 0.0, 0, 0
        accum       = self.cfg.gradient_accumulation_steps
        gate_alpha  = self._gate_alpha(epoch)
        use_seg     = self.cfg.use_seg

        self.optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(self.train_loader,
                    desc=f"Train {epoch}/{self.cfg.epochs}", leave=False)

        for i, batch in enumerate(pbar, start=1):
            x      = batch["image"].to(self.cfg.device, non_blocking=True)
            labels = batch["label"].to(self.cfg.device, non_blocking=True)
            masks  = batch["mask"]   # list; kept on CPU until loss computation

            with get_autocast_ctx(self.cfg.use_amp):
                if use_seg:
                    cls_logits, seg_logits = self.model(
                        x, gate_alpha=gate_alpha, return_seg=True
                    )
                else:
                    cls_logits = self.model(x, gate_alpha=gate_alpha)
                    seg_logits = None

                loss, log = multitask_loss(
                    cls_logits      = cls_logits,
                    seg_logits      = seg_logits,
                    labels          = labels,
                    masks           = masks,
                    lambda_cls      = self.cfg.lambda_cls,
                    lambda_seg      = self.cfg.lambda_seg,
                    label_smoothing = self.cfg.label_smoothing,
                    device          = self.cfg.device,
                )

            correct    += (cls_logits.argmax(dim=1) == labels).sum().item()
            self.scaler.scale(loss / accum).backward()

            do_step = (i % accum == 0) or (i == len(self.train_loader))
            if do_step:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.cfg.grad_clip_norm
                )
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                if self.ema is not None:
                    self.ema.update()
                self.optimizer.zero_grad(set_to_none=True)

            bs = x.size(0)
            total_loss += loss.item() * bs
            n          += bs

            if i % self.cfg.log_interval == 0:
                lr = self.optimizer.param_groups[0]["lr"]
                pbar.set_postfix(
                    loss=f"{loss.item():.4f}",
                    lr=f"{lr:.2e}",
                    ga=f"{gate_alpha:.2f}",
                    seg_n=log["n_seg"],
                )

        return {"loss": total_loss / max(1, n), "acc1": correct / max(1, n)}

    # ── Evaluation ────────────────────────────────────────────────────────────

    @torch.no_grad()
    def evaluate(self, loader, use_ema=False,
                 return_arrays=False, compute_auc=True) -> Dict:
        if use_ema and self.ema is not None:
            self.ema.apply_shadow()
        self.model.eval()

        all_preds, all_targets = [], []
        all_probs = [] if compute_auc else None
        total_loss, correct, n = 0.0, 0, 0

        for batch in tqdm(loader, desc="Eval", leave=False):
            x      = batch["image"].to(self.cfg.device, non_blocking=True)
            labels = batch["label"].to(self.cfg.device, non_blocking=True)

            with get_autocast_ctx(self.cfg.use_amp):
                # Always classification-only during eval (no seg head needed)
                cls_logits = self.model(x, gate_alpha=1.0, return_seg=False)
                loss = F.cross_entropy(cls_logits, labels,
                                        label_smoothing=self.cfg.label_smoothing)

            probs  = F.softmax(cls_logits.float(), dim=1)
            preds  = probs.argmax(dim=1)
            correct   += (preds == labels).sum().item()
            bs         = x.size(0)
            total_loss += loss.item() * bs
            n          += bs

            all_preds.extend(preds.cpu().numpy().tolist())
            all_targets.extend(labels.cpu().numpy().tolist())
            if compute_auc and all_probs is not None:
                all_probs.append(probs.cpu().numpy())

        if use_ema and self.ema is not None:
            self.ema.restore()

        preds_np   = np.array(all_preds)
        targets_np = np.array(all_targets)

        macro_auc = -1.0
        probs_np  = None
        if compute_auc and all_probs:
            probs_np = np.concatenate(all_probs, 0).astype(np.float64)
            try:
                macro_auc = roc_auc_score(
                    targets_np, probs_np,
                    multi_class="ovr", average="macro",
                    labels=np.arange(self.cfg.num_classes),
                )
            except Exception as e:
                print(f"⚠️ AUC failed: {e}")

        out = {
            "loss":            float(total_loss / max(1, n)),
            "acc1":            float(correct / max(1, n)),
            "macro_f1":        float(f1_score(targets_np, preds_np, average="macro")),
            "micro_f1":        float(f1_score(targets_np, preds_np, average="micro")),
            "weighted_f1":     float(f1_score(targets_np, preds_np, average="weighted")),
            "macro_precision": float(precision_score(targets_np, preds_np,
                                                      average="macro", zero_division=0)),
            "macro_recall":    float(recall_score(targets_np, preds_np,
                                                   average="macro", zero_division=0)),
            "macro_auc":       float(macro_auc),
        }
        if return_arrays:
            out["preds"]   = preds_np
            out["targets"] = targets_np
            if probs_np is not None:
                out["probs"] = probs_np
        return out

    # ── Checkpoint helpers ────────────────────────────────────────────────────

    def _save_checkpoint(self, epoch, is_best):
        base = self._base_model()
        ckpt = {
            "epoch":       int(epoch),
            "model":       base.state_dict(),
            "optimizer":   self.optimizer.state_dict(),
            "scheduler":   self.scheduler.state_dict(),
            "scaler":      self.scaler.state_dict(),
            "config":      self.cfg.__dict__,
            "best_val_f1": float(self.best_val_f1),
            "bad_epochs":  int(self.bad_epochs),
            "train_log":   self.train_log,
        }
        if self.ema is not None:
            ckpt["ema_shadow"] = {
                k: v.detach().clone().cpu() for k, v in self.ema.shadow.items()
            }
        self.ckpt_manager.save_checkpoint(
            ckpt, epoch, is_best,
            save_epoch_ckpt=self.cfg.save_epoch_checkpoints,
            epoch_interval=self.cfg.save_epoch_every,
            keep_n=self.cfg.keep_last_n_checkpoints,
        )

    def _save_train_log(self):
        if not self.train_log:
            return
        with open(self.logs_dir / "train_log.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=self.train_log[0].keys())
            w.writeheader(); w.writerows(self.train_log)

    def resume_from(self, ckpt_path: Path):
        print(f"\n📂 Resuming from: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=self.cfg.device)
        load_state_dict_robust(self._base_model(), ckpt["model"])
        for name, obj in [("optimizer", self.optimizer),
                           ("scheduler", self.scheduler),
                           ("scaler",    self.scaler)]:
            if name in ckpt:
                try:
                    obj.load_state_dict(ckpt[name])
                except Exception as e:
                    print(f"⚠️ {name} load failed: {e}")
        if self.ema is not None and "ema_shadow" in ckpt:
            for k, v in ckpt["ema_shadow"].items():
                self.ema.shadow[k] = v.to(self.cfg.device, non_blocking=True)
        self.best_val_f1 = float(ckpt.get("best_val_f1", -1.0))
        self.bad_epochs  = int(ckpt.get("bad_epochs", 0))
        self.start_epoch = max(1, int(ckpt.get("epoch", 0)) + 1)
        if "train_log" in ckpt:
            self.train_log = ckpt["train_log"]
        print(f"✅ Resumed: next epoch {self.start_epoch}, best F1 {self.best_val_f1:.4f}")

    # ── Main training loop ────────────────────────────────────────────────────

    def fit(self):
        print("\n" + "=" * 80)
        print("START TRAINING — SwinTCCA")
        print("=" * 80)
        print(f"Experiment : {self.cfg.experiment_name}")
        print(f"use_hsv    : {self.cfg.use_hsv}")
        print(f"use_seg    : {self.cfg.use_seg}")
        print(f"use_ema    : {self.cfg.use_ema}")
        print(f"Params     : {self.params_m:.2f}M")
        print("=" * 80)

        eff_ema_val = False

        for epoch in range(self.start_epoch, self.cfg.epochs + 1):
            train_m   = self.train_one_epoch(epoch)
            use_ema_v = self.cfg.use_ema and (self.ema is not None)
            val_m     = self.evaluate(self.val_loader, use_ema=use_ema_v,
                                       return_arrays=False,
                                       compute_auc=self.cfg.compute_val_auc)

            lr  = float(self.optimizer.param_groups[0]["lr"])
            tag = " (EMA)" if use_ema_v else ""
            print(f"\nEpoch {epoch}/{self.cfg.epochs} | LR: {lr:.2e}")
            print(f"  Train  Loss: {train_m['loss']:.4f}  Acc@1: {train_m['acc1']:.4f}")
            print(f"  Val{tag} Loss: {val_m['loss']:.4f}  Acc@1: {val_m['acc1']:.4f}  "
                  f"Macro-F1: {val_m['macro_f1']:.4f}")

            self.train_log.append({
                "epoch": int(epoch), "lr": float(lr),
                "train_loss": float(train_m["loss"]),
                "train_acc1": float(train_m["acc1"]),
                "val_loss":   float(val_m["loss"]),
                "val_acc1":   float(val_m["acc1"]),
                "val_macro_f1": float(val_m["macro_f1"]),
                "val_micro_f1": float(val_m["micro_f1"]),
                "val_macro_auc": float(val_m["macro_auc"]),
            })
            self._save_train_log()

            improved = float(val_m["macro_f1"]) > self.best_val_f1
            if improved:
                eff_ema_val      = use_ema_v
                self.best_val_f1 = float(val_m["macro_f1"])
                self.bad_epochs  = 0
                self._save_checkpoint(epoch, is_best=True)
                print(f"  🌟 New best Macro-F1: {self.best_val_f1:.4f}")
            else:
                self.bad_epochs += 1
                self._save_checkpoint(epoch, is_best=False)
                print(f"  No improvement "
                      f"({self.bad_epochs}/{self.cfg.early_stopping_patience})")

            if self.bad_epochs >= self.cfg.early_stopping_patience:
                print("\n⏹️ Early stopping triggered.")
                break

        # ── Final test evaluation ─────────────────────────────────────────
        print("\n" + "=" * 80)
        print("FINAL TEST EVALUATION")
        print("=" * 80)

        best_path = self.ckpt_manager.temp_dir / "best_model.pth"
        if not best_path.exists():
            raise FileNotFoundError(f"No best_model.pth in {self.ckpt_manager.temp_dir}")

        best_ckpt  = torch.load(best_path, map_location=self.cfg.device)
        base_model = self._base_model()

        if self.cfg.use_ema and "ema_shadow" in best_ckpt:
            print(f"📊 Loading EMA weights (epoch {best_ckpt.get('epoch','?')})")
            state = dict(best_ckpt["model"])
            for k, v in best_ckpt["ema_shadow"].items():
                if k in state:
                    state[k] = v.to(self.cfg.device, non_blocking=True)
            load_state_dict_robust(base_model, state)
            used_ema = True
        else:
            load_state_dict_robust(base_model, best_ckpt["model"])
            used_ema = False

        test_m = self.evaluate(self.test_loader, use_ema=False,
                                return_arrays=True, compute_auc=True)
        test_m["used_ema_weights"] = used_ema
        test_m["params_m"]         = float(self.params_m)

        print(f"\n📊 Test Results:")
        print(f"   Acc@1    : {test_m['acc1']:.4f}")
        print(f"   Macro-F1 : {test_m['macro_f1']:.4f}")
        auc = test_m["macro_auc"]
        print(f"   Macro-AUC: {auc:.4f}" if auc >= 0 else "   Macro-AUC: N/A")
        print(f"   Params   : {test_m['params_m']:.2f}M")

        # Save artifacts using CheckpointManager
        self.ckpt_manager.copy_final_artifacts(
            best_ckpt_path = best_path,
            config         = self.cfg,
            train_log      = self.train_log,
            test_metrics   = test_m,
            class_to_idx   = {c: i for i, c in enumerate(TeaLeafDataset.CLASS_NAMES)},
            classes        = TeaLeafDataset.CLASS_NAMES,
            logs_dir       = self.logs_dir,
        )
        return test_m


# =============================================================================
# CLI
# =============================================================================

def parse_args():
    p = argparse.ArgumentParser(
        description="Train Swin-S + TCCA (Phases 3a/3b/3c)",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--exp_name",  type=str, required=True)
    p.add_argument("--run_dir",   type=str,
                   default="/kaggle/working/experiments_tcca")
    p.add_argument("--data_root", type=str,
                   default="/kaggle/input/tea-leaf701515/"
                           "tea_leaf_processed_dataset/tea_leaf_processed_dataset")
    p.add_argument("--mask_dir",  type=str, default="",
                   help="Path to pseudo-mask directory. "
                        "Required when --use_seg is set.")

    p.add_argument("--use_tcca",  action="store_true",
                   help="Enable TCCA (should always be True for this script).")
    p.add_argument("--use_hsv",   action="store_true",
                   help="Enable HSV color branch.")
    p.add_argument("--hsv_raw",   action="store_true",
                   help="Use raw HSV (3-ch). Default: sin/cos (4-ch).")
    p.add_argument("--use_seg",   action="store_true",
                   help="Enable FPN segmentation head (Phase 3c).")

    p.add_argument("--pretrained",         action="store_true")
    p.add_argument("--drop_path_rate",     type=float, default=0.2)
    p.add_argument("--batch_size",         type=int,   default=48)
    p.add_argument("--epochs",             type=int,   default=80)
    p.add_argument("--lr",                 type=float, default=5e-4)
    p.add_argument("--num_workers",        type=int,   default=2)
    p.add_argument("--seed",               type=int,   default=42)
    p.add_argument("--use_ema",            action="store_true")
    p.add_argument("--gate_warmup_epochs", type=int,   default=5)
    p.add_argument("--lambda_seg",         type=float, default=0.5)
    p.add_argument("--gradient_accumulation_steps", type=int, default=1)
    p.add_argument("--compute_val_auc",    action="store_true")
    p.add_argument("--ckpt_temp_dir",      type=str,   default="/kaggle/working/temp")
    p.add_argument("--resume",             type=str,   default=None)
    p.add_argument("--auto_resume",        action="store_true")
    p.add_argument("--save_epoch_checkpoints", action="store_true")
    p.add_argument("--no_dp",             action="store_true")

    return p.parse_args()


def main():
    args = parse_args()

    cfg = TCCAConfig(
        pretrained             = args.pretrained,
        drop_path_rate         = args.drop_path_rate,
        use_hsv                = args.use_hsv,
        hsv_use_sincos         = (not args.hsv_raw),
        use_seg                = args.use_seg,
        data_root              = args.data_root,
        mask_dir               = args.mask_dir,
        batch_size             = args.batch_size,
        epochs                 = args.epochs,
        lr                     = args.lr,
        num_workers            = args.num_workers,
        seed                   = args.seed,
        use_ema                = args.use_ema,
        gate_warmup_epochs     = args.gate_warmup_epochs,
        lambda_seg             = args.lambda_seg,
        gradient_accumulation_steps = args.gradient_accumulation_steps,
        compute_val_auc        = args.compute_val_auc,
        run_dir                = args.run_dir,
        experiment_name        = args.exp_name,
        ckpt_temp_dir          = args.ckpt_temp_dir,
        save_epoch_checkpoints = args.save_epoch_checkpoints,
        use_data_parallel      = (not args.no_dp),
    )

    print("=" * 80)
    print(f"CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"PyTorch: {torch.__version__}  |  timm: {timm.__version__}")
    print("=" * 80)

    trainer = TCCATrainer(cfg)

    if args.resume:
        trainer.resume_from(Path(args.resume))
    elif args.auto_resume:
        last = trainer.ckpt_manager.temp_dir / "last_checkpoint.pth"
        if last.exists():
            trainer.resume_from(last)

    trainer.fit()


if __name__ == "__main__":
    main()

Overwriting train_tcca.py


**HSV-sincos + TCCA + EMA**

In [1]:
!python train_tcca.py \
    --exp_name swin_s_hsv_sincos_tcca_ema_seed42 \
    --use_hsv --use_ema --pretrained --seed 42 \
    --no_dp --gradient_accumulation_steps 2

✅ Imported utilities from train_swin_three_models.py
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.9.0+cu126  |  timm: 1.0.24

📁 Directories:
   Temp:  /kaggle/temp/swin_s_hsv_sincos_tcca_ema_seed42
   Final: /kaggle/working/experiments_tcca/swin_s_hsv_sincos_tcca_ema_seed42

🏗️  Building SwinTCCA
   use_hsv=True  use_seg=False  use_ema=True
model.safetensors: 100%|█████████████████████| 200M/200M [00:03<00:00, 61.4MB/s]
   Params: 52.75M

📦 Data: Train=6091 | Val=851 | Test=852

START TRAINING — SwinTCCA
Experiment : swin_s_hsv_sincos_tcca_ema_seed42
use_hsv    : True
use_seg    : False
use_ema    : True
Params     : 52.75M
                                                                                
Epoch 1/80 | LR: 1.01e-04
  Train  Loss: 1.5038  Acc@1: 0.4886
  Val (EMA) Loss: 2.3057  Acc@1: 0.1857  Macro-F1: 0.0610
  🌟 New best Macro-F1: 0.0610
                                                                                
Epoch 2/80 | LR: 2.01e-04
  Train  Loss: 0

In [7]:
!python train_tcca.py \
    --exp_name swin_s_hsv_sincos_tcca_ema_seed1337 \
    --use_hsv --use_ema --pretrained --seed 1337 \
    --no_dp --gradient_accumulation_steps 2

✅ Imported utilities from train_swin_three_models.py
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.10.0+cu128  |  timm: 1.0.25

📁 Directories:
   Temp:  /kaggle/working/temp/swin_s_hsv_sincos_tcca_ema_seed1337
   Final: /kaggle/working/experiments_tcca/swin_s_hsv_sincos_tcca_ema_seed1337

🏗️  Building SwinTCCA
   use_hsv=True  use_seg=False  use_ema=True
   Params: 52.75M

📦 Data: Train=6091 | Val=851 | Test=852

START TRAINING — SwinTCCA
Experiment : swin_s_hsv_sincos_tcca_ema_seed1337
use_hsv    : True
use_seg    : False
use_ema    : True
Params     : 52.75M
                                                                                
Epoch 1/80 | LR: 1.01e-04
  Train  Loss: 1.4332  Acc@1: 0.5150
  Val (EMA) Loss: 1.9742  Acc@1: 0.1763  Macro-F1: 0.0785
  🌟 New best Macro-F1: 0.0785
                                                                                
Epoch 2/80 | LR: 2.01e-04
  Train  Loss: 0.7712  Acc@1: 0.8655
  Val (EMA) Loss: 1.9596  Acc@1: 0.1857  Macr

In [8]:
!python train_tcca.py \
    --exp_name swin_s_hsv_sincos_tcca_ema_seed2026 \
    --use_hsv --use_ema --pretrained --seed 2026 \
    --no_dp --gradient_accumulation_steps 2

✅ Imported utilities from train_swin_three_models.py
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.10.0+cu128  |  timm: 1.0.25

📁 Directories:
   Temp:  /kaggle/working/temp/swin_s_hsv_sincos_tcca_ema_seed2026
   Final: /kaggle/working/experiments_tcca/swin_s_hsv_sincos_tcca_ema_seed2026

🏗️  Building SwinTCCA
   use_hsv=True  use_seg=False  use_ema=True
   Params: 52.75M

📦 Data: Train=6091 | Val=851 | Test=852

START TRAINING — SwinTCCA
Experiment : swin_s_hsv_sincos_tcca_ema_seed2026
use_hsv    : True
use_seg    : False
use_ema    : True
Params     : 52.75M
                                                                                
Epoch 1/80 | LR: 1.01e-04
  Train  Loss: 1.5264  Acc@1: 0.4669
  Val (EMA) Loss: 2.2264  Acc@1: 0.2127  Macro-F1: 0.0860
  🌟 New best Macro-F1: 0.0860
                                                                                
Epoch 2/80 | LR: 2.01e-04
  Train  Loss: 0.7696  Acc@1: 0.8637
  Val (EMA) Loss: 2.2013  Acc@1: 0.2150  Macr

**HSV-sincos + TCCA + Seg + EMA**

In [3]:
!python train_tcca.py \
    --exp_name swin_s_hsv_sincos_tcca_seg_ema_seed42 \
    --use_hsv --use_seg --use_ema --pretrained --seed 42 \
    --no_dp --gradient_accumulation_steps 2 \
    --mask_dir /kaggle/working/pseudo_masks/train

✅ Imported utilities from train_swin_three_models.py
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.10.0+cu128  |  timm: 1.0.25

📁 Directories:
   Temp:  /kaggle/working/temp/swin_s_hsv_sincos_tcca_seg_ema_seed42
   Final: /kaggle/working/experiments_tcca/swin_s_hsv_sincos_tcca_seg_ema_seed42

🏗️  Building SwinTCCA
   use_hsv=True  use_seg=True  use_ema=True
model.safetensors: 100%|█████████████████████| 200M/200M [00:05<00:00, 33.8MB/s]
   Params: 53.00M

📦 Data: Train=6091 | Val=851 | Test=852
   Pseudo-masks: 6091

START TRAINING — SwinTCCA
Experiment : swin_s_hsv_sincos_tcca_seg_ema_seed42
use_hsv    : True
use_seg    : True
use_ema    : True
Params     : 53.00M
                                                                                
Epoch 1/80 | LR: 1.01e-04
  Train  Loss: 2.1924  Acc@1: 0.4815
  Val (EMA) Loss: 2.2682  Acc@1: 0.2162  Macro-F1: 0.0519
  🌟 New best Macro-F1: 0.0519
                                                                                
E

In [3]:
!python train_tcca.py \
    --exp_name swin_s_hsv_sincos_tcca_seg_ema_seed1337 \
    --use_hsv --use_seg --use_ema --pretrained --seed 1337 \
    --no_dp --gradient_accumulation_steps 2 \
    --mask_dir /kaggle/working/pseudo_masks/train

✅ Imported utilities from train_swin_three_models.py
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.10.0+cu128  |  timm: 1.0.25

📁 Directories:
   Temp:  /kaggle/working/temp/swin_s_hsv_sincos_tcca_seg_ema_seed1337
   Final: /kaggle/working/experiments_tcca/swin_s_hsv_sincos_tcca_seg_ema_seed1337

🏗️  Building SwinTCCA
   use_hsv=True  use_seg=True  use_ema=True
model.safetensors: 100%|█████████████████████| 200M/200M [00:02<00:00, 87.5MB/s]
   Params: 53.00M

📦 Data: Train=6091 | Val=851 | Test=852
   Pseudo-masks: 6091

START TRAINING — SwinTCCA
Experiment : swin_s_hsv_sincos_tcca_seg_ema_seed1337
use_hsv    : True
use_seg    : True
use_ema    : True
Params     : 53.00M
                                                                                
Epoch 1/80 | LR: 1.01e-04
  Train  Loss: 2.1879  Acc@1: 0.4741
  Val (EMA) Loss: 2.3531  Acc@1: 0.0987  Macro-F1: 0.0528
  🌟 New best Macro-F1: 0.0528
                                                                            

In [4]:
!python train_tcca.py \
    --exp_name swin_s_hsv_sincos_tcca_seg_ema_seed2026 \
    --use_hsv --use_seg --use_ema --pretrained --seed 2026 \
    --no_dp --gradient_accumulation_steps 2 \
    --mask_dir /kaggle/working/pseudo_masks/train

✅ Imported utilities from train_swin_three_models.py
CUDA: True
  GPU 0: Tesla T4
  GPU 1: Tesla T4
PyTorch: 2.10.0+cu128  |  timm: 1.0.25

📁 Directories:
   Temp:  /kaggle/working/temp/swin_s_hsv_sincos_tcca_seg_ema_seed2026
   Final: /kaggle/working/experiments_tcca/swin_s_hsv_sincos_tcca_seg_ema_seed2026

🏗️  Building SwinTCCA
   use_hsv=True  use_seg=True  use_ema=True
   Params: 53.00M

📦 Data: Train=6091 | Val=851 | Test=852
   Pseudo-masks: 6091

START TRAINING — SwinTCCA
Experiment : swin_s_hsv_sincos_tcca_seg_ema_seed2026
use_hsv    : True
use_seg    : True
use_ema    : True
Params     : 53.00M
                                                                                
Epoch 1/80 | LR: 1.01e-04
  Train  Loss: 2.0811  Acc@1: 0.5111
  Val (EMA) Loss: 2.0828  Acc@1: 0.0987  Macro-F1: 0.0520
  🌟 New best Macro-F1: 0.0520
                                                                                
Epoch 2/80 | LR: 2.01e-04
  Train  Loss: 1.3747  Acc@1: 0.8637
  Val (EMA) L